# Quantum MRI encoding information and robustness study

This notebook implements the four directions in **bqml_3more_innovations.docx** for
the [specified Kaggle dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset/data).
It is self-contained: no starter ZIP, external code repository or quantum SDK is required.

**Start here**

1. Open this `.ipynb` in Google Colab using **File → Upload notebook**.
2. Select a GPU runtime to accelerate frozen ResNet18 image features. The quantum
   heads use a small exact CPU simulator with PyTorch autograd.
3. Run cells from top to bottom. The default `PROFILE='pilot'` runs every stage
   with reduced budgets. Use `smoke` to check the environment first. Use `full`
   for all eligible images, five seeds and the larger prespecified search.
4. Read the data audit and preprocessing examples before interpreting results.
5. Download the generated results ZIP. Save that ZIP before a Colab session ends.

**What the supplied data can establish**

The downloaded release is discovered at runtime rather than assuming a fixed image
count. The release inspected during development contained 7,200 class-folder
images. It supplied no verified patient IDs, scanner/site metadata or tumor masks.
Without original patient metadata, all inference is labeled **duplicate-group
proxy analysis**, not patient-level validation. The official `Testing` folder is
an internal benchmark partition, not an independent cohort.

Optional CSVs in the configuration activate real patient IDs, expert tumor/control
masks, and a separate compatible external cohort. Missing evidence is explicitly
reported; the notebook does not invent IDs, masks or external patients.

**Circuit scope**

The original paper's complete encoder/circuit equations were not provided. This
notebook implements fully documented **reference** passive and coherent norm
encoders, a common five-qubit circuit, and a separate four-qubit baseline. These
are new reproducible experiments, not a verified reproduction of the original
unpublished Coherent5/Passive5. No quantum hardware is used.

**Primary question**

How do norm retention, signed-direction information and circuit constraints affect
generalization and shortcut sensitivity in MRI classification?

## Cell 01 Install and check dependencies

Run once in a fresh runtime. Only missing packages are installed. A PyTorch/torchvision binary mismatch raises a clear environment error rather than silently changing the model.

In [ ]:
import importlib.util, subprocess, sys
required = {'numpy':'numpy', 'scipy':'scipy', 'pandas':'pandas', 'sklearn':'scikit-learn',
            'PIL':'Pillow', 'matplotlib':'matplotlib', 'requests':'requests', 'tqdm':'tqdm',
            'torch':'torch', 'torchvision':'torchvision'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import os, json, time, copy, math, random, hashlib, zipfile, shutil, warnings, platform
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFilter
import requests
from scipy.fft import dctn
from scipy.optimize import minimize, minimize_scalar
from scipy.special import softmax, logsumexp
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from threadpoolctl import threadpool_limits
BLAS_LIMITER=threadpool_limits(limits=min(4,os.cpu_count() or 1))
from tqdm.auto import tqdm
try:
    import torch
    from torch import nn
    from torch.nn import functional as F
    import torchvision
    from torchvision import models, transforms
except Exception as exc:
    raise RuntimeError('PyTorch/torchvision import failed. In Colab, reconnect to a fresh standard runtime; do not mix CPU and CUDA wheel builds.') from exc
try:
    from IPython.display import display, Markdown
except ImportError:
    display = print
    Markdown = str
print('Python', platform.python_version(), '| torch', torch.__version__, '| torchvision', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none; CPU execution is supported')

## Cell 02 Configuration

Choose the profile before running experiments. `pilot` is a complete pipeline check with reduced image and fitting budgets; its results are exploratory. `full` uses all eligible data, five seeds, three regularization choices, three signed-reference scales, five outer development folds, and 3,000 patient/proxy bootstrap resamples. Full runs take longer and can exceed a single session. Features are cached; fitted checkpoints and results are exported.

Optional inputs:

* `PATIENT_MANIFEST_CSV`: `relative_path,patient_id`. Supply verified original IDs for every image, never filename guesses. Format validation cannot establish their provenance.
* `MASK_MANIFEST_CSV`: `relative_path,tumor_mask_path,control_mask_path`. Real masks must align with original pixels. Expert controls should match area, tissue and location.
* `EXTERNAL_DATA_ROOT`: separate four-class image folders. `EXTERNAL_PATIENT_CSV` follows the patient schema. Confirm compatible sequences/labels and independently collected patients.
* `LOCAL_DATA_ZIP` / `LOCAL_DATA_ROOT`: optional manual Kaggle download if the public API is unavailable.
* `RESNET_WEIGHTS_FILE`: optional official ResNet18 ImageNet weight file for offline use.

The example `.02` equivalence margin is an engineering analysis choice, not a clinically validated threshold. Fix a justified margin before evaluating a new cohort.

In [ ]:
PROFILE = os.getenv('BQML_PROFILE', 'pilot')  # smoke, pilot, full
PROFILES = {
    'smoke': dict(seeds=[0], epochs=3, patience=3, gamma_grid=[1.0], l2_grid=[.001],
                  dev_units_per_class=20, eval_units_per_class=8, teacher_seeds=[123],
                  teacher_steps=8, bootstrap=100, permutation=99, stress_units_per_class=3),
    'pilot': dict(seeds=[0,1], epochs=35, patience=7, gamma_grid=[1.0], l2_grid=[.001,.01],
                  dev_units_per_class=180, eval_units_per_class=80, teacher_seeds=[123,456],
                  teacher_steps=100, bootstrap=1000, permutation=999, stress_units_per_class=20),
    'full': dict(seeds=[0,1,2,3,4], epochs=120, patience=15, gamma_grid=[.5,1.,2.], l2_grid=[.0001,.001,.01],
                 dev_units_per_class=None, eval_units_per_class=None, teacher_seeds=[123,456,789],
                 teacher_steps=250, bootstrap=3000, permutation=2999, stress_units_per_class=None),
}
if PROFILE not in PROFILES:
    raise ValueError('PROFILE must be smoke, pilot or full.')
CFG = copy.deepcopy(PROFILES[PROFILE])
CFG.update(dict(profile=PROFILE, dataset_handle='masoudnickparvar/brain-tumor-mri-dataset',
                split_seed=20261016, pca_components=16, qubits=5, layers=2,
                train_batch=128, image_batch=32, lr=.015, normalization='robust',
                image_size=224, phash_radius=4, dhash_radius=6, aspect_tolerance=.05,
                learning_fractions=[.10,.25,.50,1.0], norm_scales=[.5,.75,1.,1.5,2.],
                equivalence_margin=.02, run_unitary_control=True, run_learning_curves=True,
                run_teacher=True, run_image_stress=True, run_grouped_cv=True,
                cv_folds=2 if PROFILE=='smoke' else (3 if PROFILE=='pilot' else 5), auto_download_results=False))
# Paths are optional. Keep strings empty when the corresponding evidence is unavailable.
LOCAL_DATA_ROOT = os.getenv('BQML_DATA_ROOT', '')
LOCAL_DATA_ZIP = ''
PATIENT_MANIFEST_CSV = os.getenv('BQML_PATIENT_CSV', '')  # relative_path,patient_id
MASK_MANIFEST_CSV = os.getenv('BQML_MASK_CSV', '')  # relative_path,tumor_mask_path,control_mask_path
EXTERNAL_DATA_ROOT = os.getenv('BQML_EXTERNAL_ROOT', '')
EXTERNAL_PATIENT_CSV = os.getenv('BQML_EXTERNAL_PATIENT_CSV', '')
RESNET_WEIGHTS_FILE = ''  # Optional local official ResNet18 IMAGENET1K_V1 state_dict.
base = Path('/content') if Path('/content').exists() else Path.cwd()
WORK_ROOT = Path(os.getenv('BQML_WORK_ROOT', str(base/'bqml_study')))
RUN_ID = os.getenv('BQML_RUN_ID', datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+PROFILE)
RUN_DIR = WORK_ROOT/'runs'/RUN_ID
if RUN_DIR.exists() and (RUN_DIR/'evaluation_complete.json').exists():
    raise RuntimeError('This run has completed. Use a new RUN_ID to preserve evaluation history.')
for p in [RUN_DIR, WORK_ROOT/'cache', WORK_ROOT/'data', RUN_DIR/'figures', RUN_DIR/'checkpoints']:
    p.mkdir(parents=True, exist_ok=True)
CLASSES = ['glioma','meningioma','notumor','pituitary']
K = len(CLASSES)
FEATURE_DEVICE = torch.device('cuda' if torch.cuda.is_available() and os.getenv('BQML_FORCE_CPU','0')!='1' else 'cpu')
QDEVICE = torch.device('cpu')  # Tiny statevectors often benefit from avoiding many GPU kernel launches.
torch.set_num_threads(min(4, os.cpu_count() or 1))
random.seed(CFG['split_seed']); np.random.seed(CFG['split_seed']); torch.manual_seed(CFG['split_seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['split_seed'])
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
torch.use_deterministic_algorithms(True, warn_only=True)
PROGRESS = os.getenv('BQML_PROGRESS','1') == '1'
STATUS = {'patient_ids_supplied':bool(PATIENT_MANIFEST_CSV), 'tumor_masks_supplied':bool(MASK_MANIFEST_CSV),
          'external_supplied':bool(EXTERNAL_DATA_ROOT), 'original_circuits_reproduced':False,
          'hardware_execution':False, 'profile':PROFILE}
VERSIONS = {n:importlib.import_module(m).__version__ for n,m in
            [('numpy','numpy'),('scipy','scipy'),('pandas','pandas'),('sklearn','sklearn'),('torch','torch'),('torchvision','torchvision')]}
def save_json(name, obj):
    path=RUN_DIR/name; path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(obj,indent=2,default=lambda v: v.item() if isinstance(v,np.generic) else str(v))+'\n')
def save_table(name, rows):
    frame=rows if isinstance(rows,pd.DataFrame) else pd.DataFrame(rows)
    if frame.empty and not len(frame.columns):frame=pd.DataFrame(columns=['status','reason'])
    frame.to_csv(RUN_DIR/name,index=False)
    return frame
save_json('configuration.json', {'config':CFG, 'classes':CLASSES, 'versions':VERSIONS,
                               'paths':{'patient_manifest':PATIENT_MANIFEST_CSV,'mask_manifest':MASK_MANIFEST_CSV,
                                        'external_root':EXTERNAL_DATA_ROOT}, 'status':STATUS})
print('Profile:',PROFILE,'| feature device:',FEATURE_DEVICE,'| outputs:',RUN_DIR)
if PROFILE!='full': print('This profile uses reduced data/optimization budgets. Use full for the prespecified complete study.')

## Cell 03 Download the exact Kaggle dataset or use a local copy

Downloads from Kaggle's public dataset endpoint, checks ZIP integrity and prevents archive path traversal. If Kaggle requires authentication or rate-limits the request, upload the dataset ZIP and set `LOCAL_DATA_ZIP`; no credentials are printed or embedded.

In [ ]:
def safe_extract(archive, destination):
    destination=Path(destination); destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for info in z.infolist():
            target=(destination/info.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError('Unsafe archive member.')
        z.extractall(destination)
def obtain_dataset():
    if LOCAL_DATA_ROOT:
        root=Path(LOCAL_DATA_ROOT)
        if not root.is_dir(): raise FileNotFoundError(f'Data root does not exist: {root}')
        return root
    archive=Path(LOCAL_DATA_ZIP) if LOCAL_DATA_ZIP else WORK_ROOT/'data'/'brain_tumor_mri.zip'
    if not archive.exists():
        url='https://www.kaggle.com/api/v1/datasets/download/'+CFG['dataset_handle']
        temp=archive.with_suffix('.partial'); archive.parent.mkdir(parents=True,exist_ok=True)
        try:
            with requests.get(url,stream=True,timeout=(20,180)) as r:
                r.raise_for_status()
                with temp.open('wb') as f, tqdm(total=int(r.headers.get('Content-Length',0)),unit='B',unit_scale=True,disable=not PROGRESS) as bar:
                    for chunk in r.iter_content(2**20):
                        if chunk: f.write(chunk); bar.update(len(chunk))
            if not zipfile.is_zipfile(temp):
                raise RuntimeError('Kaggle returned a non-ZIP response, possibly a sign-in/rate-limit page.')
            temp.replace(archive)
        except Exception as exc:
            if temp.exists(): temp.unlink()
            raise RuntimeError('Dataset download failed. Download the ZIP from the supplied Kaggle page, upload it to Colab, set LOCAL_DATA_ZIP, and rerun this cell.') from exc
    if not zipfile.is_zipfile(archive): raise ValueError('LOCAL_DATA_ZIP is not a valid ZIP archive.')
    root=WORK_ROOT/'data'/'brain_tumor_mri'
    if not (root/'.extracted_complete').exists():
        safe_extract(archive,root); (root/'.extracted_complete').write_text('complete')
    save_json('dataset_download.json', {'handle':CFG['dataset_handle'],
        'zip_sha256':hashlib.sha256(archive.read_bytes()).hexdigest(),'bytes':archive.stat().st_size})
    return root
DATA_ROOT=obtain_dataset()
print('Dataset root:',DATA_ROOT)

## Cell 04 Image inventory and exact/perceptual hashing

Decode all source images, retain original labels/splits and hash decoded RGB pixels plus shape. Record corrupt files. Compute 64-bit DCT perceptual and difference hashes. The model never receives filenames, folder labels or hashes as image features.

In [ ]:
def canonical_label(name):
    token=''.join(c for c in str(name).lower() if c.isalnum())
    aliases={'glioma':'glioma','gliomatumor':'glioma','meningioma':'meningioma','meningiomatumor':'meningioma',
             'notumor':'notumor','normal':'notumor','pituitary':'pituitary','pituitarytumor':'pituitary'}
    return aliases.get(token)
def read_gray(path):
    with Image.open(path) as im:
        return np.asarray(ImageOps.exif_transpose(im).convert('L'),dtype=np.float64)/255.
def hashes_and_metadata(path):
    with Image.open(path) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        rgb=np.asarray(im); width,height=im.size
        exact=hashlib.sha256(np.asarray(rgb.shape,dtype=np.int64).tobytes()+rgb.tobytes()).hexdigest()
        gray=im.convert('L')
        small=np.asarray(gray.resize((32,32),Image.Resampling.LANCZOS),dtype=float)
        low=dctn(small,norm='ortho')[:8,:8].ravel(); bits=low>np.median(low[1:]); bits[0]=False
        ph=sum(int(bit)<<i for i,bit in enumerate(bits))
        dhim=np.asarray(gray.resize((9,8),Image.Resampling.BILINEAR))
        dh=sum(int(bit)<<i for i,bit in enumerate((dhim[:,1:]>dhim[:,:-1]).ravel()))
        return exact,ph,dh,width,height
def scan_images(root):
    root=Path(root); records=[]; failures=[]
    paths=sorted(p for p in root.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'})
    for path in tqdm(paths,desc='Decode and hash images',disable=not PROGRESS):
        rel=path.relative_to(root).as_posix()
        label=next((canonical_label(part) for part in reversed(path.relative_to(root).parts[:-1]) if canonical_label(part)),None)
        if label is None: continue
        split='Testing' if any(s.lower() in {'testing','test'} for s in path.relative_to(root).parts[:-1]) else 'Training'
        try:
            exact,ph,dh,w,h=hashes_and_metadata(path)
            records.append(dict(relative_path=rel,path=str(path),label=label,y=CLASSES.index(label),original_split=split,
                                pixel_sha256=exact,phash=f'{ph:016x}',dhash=f'{dh:016x}',width=w,height=h))
        except Exception as exc: failures.append(dict(relative_path=rel,error=type(exc).__name__+': '+str(exc)))
    frame=pd.DataFrame(records)
    if frame.empty or set(frame.label)!=set(CLASSES):
        raise ValueError('Could not locate all four class folders. Check DATA_ROOT and CLASSES.')
    frame['sample_id']=frame.relative_path
    return frame,failures
raw_manifest,decode_failures=scan_images(DATA_ROOT)
save_table('raw_manifest.csv',raw_manifest); save_table('decode_failures.csv',pd.DataFrame(decode_failures,columns=['relative_path','error']))
display(pd.crosstab(raw_manifest.label,raw_manifest.original_split))
print('Decoded images:',len(raw_manifest),'| failures:',len(decode_failures))
print('Counts are discovered from the downloaded release; no fixed 7,023/7,200 assumption is used.')

## Cell 05 Duplicate graph and verified patient metadata

Exact copies and prespecified near candidates form a graph. The near rule requires pHash distance ≤4, dHash distance ≤6 and aspect-ratio log difference ≤0.05. Graph connectivity can overmerge similar images; all edges and quarantined components are exported for review. It is a conservative heuristic, not verified patient linkage.

Cross-label components are quarantined without relabeling. Exact duplicate representatives prioritize Testing. Training images linked to Testing groups are excluded from fitting. If supplied, original patient IDs also join the graph so no patient crosses a partition. Verify the effect of the audit before publication.

In [ ]:
class UnionFind:
    def __init__(self,n): self.parent=list(range(n)); self.rank=[0]*n
    def find(self,i):
        while self.parent[i]!=i:
            self.parent[i]=self.parent[self.parent[i]]; i=self.parent[i]
        return i
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.rank[a]<self.rank[b]:a,b=b,a
        self.parent[b]=a
        if self.rank[a]==self.rank[b]:self.rank[a]+=1
class BKTree:
    def __init__(self): self.root=None
    def add(self,value,index):
        if self.root is None:self.root=[value,[index],{}];return
        node=self.root
        while True:
            d=(value^node[0]).bit_count()
            if d==0:node[1].append(index);return
            if d not in node[2]:node[2][d]=[value,[index],{}];return
            node=node[2][d]
    def query(self,value,radius):
        found=[]; stack=[] if self.root is None else [self.root]
        while stack:
            node=stack.pop(); d=(value^node[0]).bit_count()
            if d<=radius:found.extend(node[1])
            stack.extend(child for dist,child in node[2].items() if d-radius<=dist<=d+radius)
        return found
def normalize_relative(v):return str(v).replace('\\','/').removeprefix('./')
def attach_patients(frame,csv_path):
    frame=frame.copy()
    if not csv_path:
        frame['patient_id']=''; return frame,False
    meta=pd.read_csv(csv_path,dtype=str).fillna('')
    if not {'relative_path','patient_id'}<=set(meta.columns):
        raise ValueError('Patient CSV requires relative_path and patient_id.')
    meta.relative_path=meta.relative_path.map(normalize_relative)
    if meta.relative_path.duplicated().any():raise ValueError('Repeated image paths in patient metadata.')
    frame=frame.merge(meta[['relative_path','patient_id']],on='relative_path',how='left',validate='one_to_one')
    if frame.patient_id.isna().any() or frame.patient_id.str.strip().eq('').any():
        raise ValueError('The supplied patient manifest must cover every retained image; no inferred patient IDs are used.')
    if (frame.groupby('patient_id').y.nunique()>1).any():
        raise ValueError('A patient has multiple classes. Define a time-specific target before using this workflow.')
    return frame,True
def group_and_clean(frame,patient_csv=''):
    frame,verified=attach_patients(frame,patient_csv); frame=frame.reset_index(drop=True)
    uf=UnionFind(len(frame)); exact_edges=[]; near_edges=[]
    for _,inds in frame.groupby('pixel_sha256',sort=False).groups.items():
        inds=list(inds)
        for j in inds[1:]: uf.union(inds[0],j); exact_edges.append((inds[0],j))
    tree=BKTree(); ph=[int(s,16) for s in frame.phash]; dh=[int(s,16) for s in frame.dhash]
    aspects=(frame.width/frame.height).to_numpy()
    for i in tqdm(range(len(frame)),desc='Near-duplicate graph',disable=not PROGRESS):
        for j in tree.query(ph[i],CFG['phash_radius']):
            if (dh[i]^dh[j]).bit_count()<=CFG['dhash_radius'] and abs(math.log(aspects[i]/aspects[j]))<=CFG['aspect_tolerance']:
                uf.union(i,j); near_edges.append((j,i))
        tree.add(ph[i],i)
    if verified:
        for _,inds in frame.groupby('patient_id',sort=False).groups.items():
            inds=list(inds)
            for j in inds[1:]:uf.union(inds[0],j)
    components={}
    for i in range(len(frame)):components.setdefault(uf.find(i),[]).append(i)
    gid={root:'group_'+hashlib.sha256('|'.join(sorted(frame.loc[inds,'relative_path'])).encode()).hexdigest()[:16]
         for root,inds in components.items()}
    frame['split_group']=[gid[uf.find(i)] for i in range(len(frame))]
    mixed=frame.groupby('split_group').y.nunique()
    ambiguous=set(mixed[mixed>1].index)
    quarantined=frame[frame.split_group.isin(ambiguous)].copy()
    clean=frame[~frame.split_group.isin(ambiguous)].copy()
    # Prespecified conservative handling: quarantine cross-label graph components;
    # retain the audit for manual review rather than silently changing labels.
    clean['_priority']=(clean.original_split!='Testing').astype(int)
    clean=clean.sort_values(['_priority','relative_path']).drop_duplicates('pixel_sha256',keep='first').drop(columns='_priority')
    test_groups=set(clean.loc[clean.original_split=='Testing','split_group'])
    crossed=clean.original_split.eq('Training') & clean.split_group.isin(test_groups)
    cross_removed=clean[crossed].copy(); clean=clean[~crossed].copy()
    clean['analysis_unit']=clean.patient_id if verified else clean.split_group
    clean=clean.sort_values('relative_path').reset_index(drop=True)
    edges=[]
    for kind,pairs in [('exact',exact_edges),('near',near_edges)]:
        for a,b in pairs:
            edges.append(dict(kind=kind,a=frame.loc[a,'relative_path'],b=frame.loc[b,'relative_path'],
                              cross_original_split=bool(frame.loc[a,'original_split']!=frame.loc[b,'original_split'])))
    audit=dict(raw_images=len(frame),retained_images=len(clean),quarantined_cross_label_images=len(quarantined),
               exact_redundant_images=int(len(frame[~frame.split_group.isin(ambiguous)])-len(frame[~frame.split_group.isin(ambiguous)].drop_duplicates('pixel_sha256'))),
               cross_boundary_training_images_removed=len(cross_removed),split_groups=clean.split_group.nunique(),
               analysis_units=clean.analysis_unit.nunique(),unit_kind='verified_patient' if verified else 'duplicate_group_proxy',
               near_edges=len(near_edges),largest_raw_graph_component=max(map(len,components.values())))
    return clean,quarantined,cross_removed,pd.DataFrame(edges,columns=['kind','a','b','cross_original_split']),audit
clean_manifest,quarantine,cross_removed,duplicate_edges,duplicate_audit=group_and_clean(raw_manifest,PATIENT_MANIFEST_CSV)
UNIT_KIND=duplicate_audit['unit_kind']; STATUS['unit_kind']=UNIT_KIND
save_table('clean_manifest_before_budget.csv',clean_manifest)
save_table('quarantined_cross_label_components.csv',quarantine)
save_table('cross_boundary_training_images_removed.csv',cross_removed)
save_table('duplicate_edges.csv',duplicate_edges); save_json('duplicate_audit.json',duplicate_audit)
display(pd.DataFrame([duplicate_audit]))
if UNIT_KIND!='verified_patient':
    print('Patient IDs are unavailable. Analysis units are duplicate groups; patient independence and patient-level generalization are NOT established.')

## Cell 06 Split development, selection, calibration and evaluation groups

Preserve the cleaned official Testing partition where available. Split development groups into 70% fit, 15% selection and 15% temperature calibration, approximately by class. Entire graph groups remain together. If no official Testing exists, create a grouped 20% internal holdout. Reduced profiles cap groups without splitting their images.

In [ ]:
def cap_groups(frame,limit,seed):
    if limit is None:return frame.copy()
    group_table=frame[['split_group','y']].drop_duplicates()
    chosen=[]; rng=np.random.default_rng(seed)
    for label,part in group_table.groupby('y'):
        ids=sorted(part.split_group); chosen.extend(rng.choice(ids,min(len(ids),limit),replace=False).tolist())
    return frame[frame.split_group.isin(chosen)].copy()
has_official_test=clean_manifest.original_split.eq('Testing').any()
if has_official_test:
    dev=cap_groups(clean_manifest[clean_manifest.original_split=='Training'],CFG['dev_units_per_class'],CFG['split_seed'])
    evaluation=cap_groups(clean_manifest[clean_manifest.original_split=='Testing'],CFG['eval_units_per_class'],CFG['split_seed']+1)
else:
    eligible=cap_groups(clean_manifest,CFG['dev_units_per_class'],CFG['split_seed'])
    tab=eligible[['split_group','y']].drop_duplicates()
    a,b=train_test_split(tab.split_group,test_size=.2,stratify=tab.y,random_state=CFG['split_seed'])
    dev=eligible[eligible.split_group.isin(a)].copy();evaluation=eligible[eligible.split_group.isin(b)].copy()
tab=dev[['split_group','y']].drop_duplicates()
if tab.groupby('y').size().min()<10:
    raise ValueError('Too few development groups/class for fit/selection/calibration. Increase the profile budget or inspect the duplicate graph.')
fit_groups,hold_groups=train_test_split(tab.split_group,test_size=.30,stratify=tab.y,random_state=CFG['split_seed'])
hold=tab[tab.split_group.isin(hold_groups)]
selection_groups,temperature_groups=train_test_split(hold.split_group,test_size=.5,stratify=hold.y,random_state=CFG['split_seed']+1)
dev['partition']=np.where(dev.split_group.isin(fit_groups),'fit',np.where(dev.split_group.isin(selection_groups),'selection','temperature'))
evaluation['partition']='evaluation'
manifest=pd.concat([dev,evaluation],ignore_index=True).sort_values('relative_path').reset_index(drop=True)
for a in ['fit','selection','temperature','evaluation']:
    if set(manifest.loc[manifest.partition==a,'y'])!=set(range(K)):
        raise ValueError(f'{a} lacks a class; revise data budgets before running models.')
for group_col in ['split_group','analysis_unit']:
    if (manifest.groupby(group_col).partition.nunique()>1).any():raise AssertionError(f'{group_col} leakage detected.')
save_table('study_manifest.csv',manifest)
SPLITS={p:np.flatnonzero(manifest.partition.eq(p).to_numpy()) for p in ['fit','selection','temperature','evaluation']}
save_json('split_manifest.json',{p:manifest.iloc[idx][['relative_path','analysis_unit','split_group','y']].to_dict('records') for p,idx in SPLITS.items()})
display(pd.crosstab(manifest.partition,manifest.label))
print('Independent external cohort:',bool(EXTERNAL_DATA_ROOT),'| official Testing is an internal benchmark partition, not an external cohort.')

## Cell 07 Preprocessing visual check

Display examples before and after the declared intensity transform. Robust normalization uses per-image 1st/99th percentiles and is followed by direct resizing to 224×224 and ImageNet channel normalization. Direct resizing avoids a center crop but can distort aspect ratio. Inspect anatomy and retain this transform for all conditions.

In [ ]:
def normalize_image(image,method=None):
    image=np.asarray(image,dtype=float); method=CFG['normalization'] if method is None else method
    if not np.isfinite(image).all():raise ValueError('Nonfinite image intensities.')
    if method=='robust':
        lo,hi=np.percentile(image,[1,99])
        image=np.clip((image-lo)/max(hi-lo,1e-8),0,1)
    elif method=='none':image=np.clip(image,0,1)
    else:raise ValueError('Unknown image normalization.')
    return Image.fromarray(np.rint(image*255).astype(np.uint8),'L').convert('RGB')
fig,axes=plt.subplots(2,4,figsize=(12,6))
for j,label in enumerate(CLASSES):
    row=manifest[manifest.label==label].iloc[0];raw=read_gray(row.path)
    axes[0,j].imshow(raw,cmap='gray');axes[1,j].imshow(normalize_image(raw))
    axes[0,j].set_title(label+' raw');axes[1,j].set_title('preprocessed')
for ax in axes.ravel():ax.axis('off')
fig.tight_layout();fig.savefig(RUN_DIR/'figures'/'preprocessing_examples.png',dpi=160);plt.show();plt.close(fig)

## Cell 08 Frozen ResNet18 feature extractor and feature cache

Use a frozen ImageNet-pretrained ResNet18 to extract 512 features. No MRI fine-tuning occurs. Cache keys include source pixel hashes, row order and preprocessing choices. A backbone fine-tuned on this cohort would need to be trained again inside every development fold; do not substitute a leaky feature cache.

In [ ]:
weights=models.ResNet18_Weights.IMAGENET1K_V1
backbone=models.resnet18(weights=None if RESNET_WEIGHTS_FILE else weights)
if RESNET_WEIGHTS_FILE:
    backbone.load_state_dict(torch.load(RESNET_WEIGHTS_FILE,map_location='cpu',weights_only=True))
backbone.fc=nn.Identity();backbone.requires_grad_(False);backbone.eval().to(FEATURE_DEVICE)
image_transform=transforms.Compose([transforms.Resize((CFG['image_size'],CFG['image_size'])),
    transforms.ToTensor(),transforms.Normalize(mean=[.485,.456,.406],std=[.229,.224,.225])])
def extract_features(frame,intervention=None,cache_tag=None):
    cache_key=hashlib.sha256(('resnet18_imagenet1k_v1_directresize_v1|'+CFG['normalization']+'|'+str(CFG['image_size'])+'|'+
        '|'.join(frame.pixel_sha256.astype(str))+'|'+('|'.join(frame.relative_path.astype(str)))).encode()).hexdigest()
    cache_path=WORK_ROOT/'cache'/('features_'+cache_key+'.npz')
    if intervention is None and cache_path.exists():
        with np.load(cache_path,allow_pickle=False) as a:
            if np.array_equal(a['sample_id'],frame.relative_path.to_numpy(dtype=str)):
                return a['X'].astype(np.float64)
    arrays=[]
    rows=list(frame.itertuples(index=False))
    for start in tqdm(range(0,len(rows),CFG['image_batch']),desc=cache_tag or 'ResNet18 features',disable=not PROGRESS):
        tensors=[]
        for row in rows[start:start+CFG['image_batch']]:
            im=read_gray(row.path)
            if intervention is not None:im=intervention(im,row)
            tensors.append(image_transform(normalize_image(im)))
        batch=torch.stack(tensors).to(FEATURE_DEVICE)
        with torch.inference_mode():result=backbone(batch).detach().cpu().numpy()
        arrays.append(result)
    x=np.concatenate(arrays).astype(np.float64)
    if not np.isfinite(x).all():raise ValueError('Nonfinite frozen features.')
    if intervention is None:
        np.savez_compressed(cache_path,X=x.astype(np.float32),sample_id=frame.relative_path.to_numpy(dtype=str))
    return x
X_ALL=extract_features(manifest)
Y_ALL=manifest.y.to_numpy(dtype=int); U_ALL=manifest.analysis_unit.to_numpy(dtype=str)
print('Frozen feature matrix:',X_ALL.shape)
save_json('feature_specification.json',dict(model='ResNet18 IMAGENET1K_V1, frozen',output_features=X_ALL.shape[1],
    resize=[CFG['image_size'],CFG['image_size']],center_crop=False,normalization=CFG['normalization']))

## Cell 09 Patient or proxy-unit aggregation and metrics

Each analysis unit gets one prediction by averaging calibrated slice probabilities. Training weights give every unit equal total weight. Report macro-F1, accuracy, balanced accuracy, AUC, NLL, Brier and ECE. ECE with small samples is descriptive. Scaler and unwhitened PCA16 fit only on training rows; they are refitted in development folds and learning-curve subsets. Unsupervised scaler/PCA statistics remain slice-weighted, which is a declared design choice.

In [ ]:
def aggregate(y,p,units):
    ids,inv=np.unique(np.asarray(units,dtype=str),return_inverse=True)
    labels=np.empty(len(ids),dtype=int);out=np.zeros((len(ids),p.shape[1]),dtype=float)
    for i in range(len(ids)):
        yi=np.unique(np.asarray(y)[inv==i])
        if len(yi)!=1:raise ValueError('An analysis unit has inconsistent labels.')
        labels[i]=yi[0]
    np.add.at(out,inv,p);out/=np.bincount(inv)[:,None]
    return ids,labels,out
def macro_f1_fast(y,pred):
    cm=np.bincount(K*np.asarray(y,dtype=int)+np.asarray(pred,dtype=int),minlength=K*K).reshape(K,K)
    denominator=cm.sum(axis=0)+cm.sum(axis=1)
    return float(np.divide(2*np.diag(cm),denominator,out=np.zeros(K,dtype=float),where=denominator>0).mean())
def unit_weights(units):
    _,inv=np.unique(np.asarray(units,dtype=str),return_inverse=True)
    w=1./np.bincount(inv)[inv]
    return w/w.mean()
def nll(y,p):return float(-np.log(np.clip(p[np.arange(len(y)),np.asarray(y,dtype=int)],1e-15,1)).mean())
def ece(y,p,bins=15):
    conf=p.max(1);correct=p.argmax(1)==y;value=0.
    for j in range(bins):
        mask=(conf>=j/bins)&(conf<(j+1)/bins if j<bins-1 else conf<=1)
        if mask.any():value+=mask.mean()*abs(float(correct[mask].mean()-conf[mask].mean()))
    return float(value)
def unit_metrics(y,p,units):
    ids,yt,pp=aggregate(y,p,units);pred=pp.argmax(1)
    cm=np.bincount(K*yt+pred,minlength=K*K).reshape(K,K)
    recall=np.divide(np.diag(cm),cm.sum(1),out=np.zeros(K,dtype=float),where=cm.sum(1)>0)
    try:auc=float(roc_auc_score(yt,pp,multi_class='ovr',labels=np.arange(K),average='macro'))
    except ValueError:auc=np.nan
    return dict(n_units=len(ids),macro_f1=macro_f1_fast(yt,pred),accuracy=float((pred==yt).mean()),
                balanced_accuracy=float(recall.mean()),nll=nll(yt,pp),ece=ece(yt,pp),
                brier=float(np.mean(np.sum((pp-np.eye(K)[yt])**2,axis=1))),auc_ovr=auc)
class FeaturePrep:
    def __init__(self,components=16):self.components=components
    def fit(self,x):
        if self.components and min(x.shape)<self.components+1:
            raise ValueError('Insufficient training rows/features for PCA16. Increase the training budget.')
        self.scaler=StandardScaler().fit(x)
        a=self.scaler.transform(x)
        self.pca=PCA(self.components,svd_solver='full').fit(a) if self.components else None
        z=self.pca.transform(a) if self.pca is not None else a
        self.r0=max(float(np.median(np.linalg.norm(z,axis=1))),1e-8)
        return self
    def transform(self,x):
        a=self.scaler.transform(x)
        return self.pca.transform(a) if self.pca is not None else a
    def save(self,path):
        np.savez_compressed(path,scaler_mean=self.scaler.mean_,scaler_scale=self.scaler.scale_,
            pca_mean=np.array([]) if self.pca is None else self.pca.mean_,
            pca_components=np.array([]) if self.pca is None else self.pca.components_,r0=self.r0)
PREP=FeaturePrep(16).fit(X_ALL[SPLITS['fit']])
PREP.save(RUN_DIR/'checkpoints'/'main_preprocessing.npz')
PCA_ALL=PREP.transform(X_ALL)

## Cell 10 Encodings and their physically available quadratic lifts

Let $r=\|x\|$, $u=x/r$, and $r_0$ be the median training norm. The common register has 32 amplitudes.

| Encoder | Prepared representation |
|---|---|
| amplitude | $[u,0]$ |
| coherent norm reference | $[\cos\theta\,u,\sin\theta\,u]$, $\theta=\arctan(r/(\gamma r_0))$ |
| passive norm reference | the coherent reference density matrix dephased across its ancilla blocks |
| signed direction | normalize $[u,\gamma,0,\ldots]$ |
| signed full | normalize $[x,\gamma r_0,0,\ldots]$ |

Amplitude encoding preserves **relative coordinate signs** but identifies $x$ and $-x$ through global phase. A norm-only ancilla retains that ambiguity unless it introduces an independent reference. The zero-vector direction uses an explicit common basis-vector convention.

The passive model's lift excludes cross-block coherences. Giving a classical comparator those missing terms would change the information available to it. Pure-state lifts have 528 unique real quadratic features; passive lifts have 272.

In [ ]:
ENCODINGS=['amplitude','passive_norm','coherent_norm','signed_direction','signed_full']
def encode_array(x,kind,r0,gamma=1.,qubits=5):
    x=np.asarray(x,dtype=float)
    if x.ndim!=2 or x.shape[1]!=16:raise ValueError('The controlled encoder requires PCA16 inputs.')
    if gamma<=0 or r0<=0:raise ValueError('Encoding scales must be positive.')
    r=np.linalg.norm(x,axis=1);u=np.divide(x,r[:,None],out=np.zeros_like(x),where=r[:,None]>1e-12)
    zero=r<=1e-12;u[zero,0]=1. # Explicit common convention for undefined zero-vector direction.
    if qubits==4:
        if kind!='amplitude':raise ValueError('Only bare amplitude uses the four-qubit reference.')
        return u
    z=np.zeros((len(x),32))
    if kind=='amplitude':z[:,:16]=u
    elif kind in {'passive_norm','coherent_norm'}:
        theta=np.arctan(r/(gamma*r0));z[:,:16]=np.cos(theta)[:,None]*u;z[:,16:]=np.sin(theta)[:,None]*u
    elif kind=='signed_direction':z[:,:16]=u;z[:,16]=gamma
    elif kind=='signed_full':z[:,:16]=x;z[:,16]=gamma*r0
    else:raise ValueError('Unknown encoding '+kind)
    return z/np.linalg.norm(z,axis=1,keepdims=True)
def lift_array(z,passive=False):
    i,j=np.triu_indices(z.shape[1]);f=z[:,i]*z[:,j]*np.where(i==j,1.,np.sqrt(2.))
    if passive:
        # Ancilla dephasing removes cross-block coherences; the classical lift must
        # not be given those unavailable terms.
        f=f[:,(i<z.shape[1]//2)==(j<z.shape[1]//2)]
    return f
def encoded_density(z,passive=False):
    rho=np.einsum('ni,nj->nij',z,z)
    if passive:
        n=z.shape[1]//2;rho[:,:n,n:]=0.;rho[:,n:,:n]=0.
    return rho
def model_inputs(x,encoding,r0,gamma=1.,kind='circuit',qubits=5):
    if kind in {'mlp','linear_raw','rbf','linear_512'}:return x
    z=encode_array(x,encoding,r0,gamma,qubits)
    return lift_array(z,encoding=='passive_norm') if kind=='quadratic' else z

## Cell 11 Exact differentiable quantum circuit and matrix-constrained controls

The shared ansatz repeats RZ then RY on every wire followed by a CNOT ring, twice. It measures wire-Z expectations and uses an affine class readout. The coherent/passive comparison shares all trainable gates and readout; only input ancilla coherence differs.

Controls include an unrestricted convex quadratic classifier on the exact lift; a rank-one factorized quadratic with a comparable parameter budget; and a dense full-unitary relaxation retaining the underlying shared observables/spectra. The full-unitary control is nonconvex and has more parameters. No control alone proves a cause of failure. The separate four-qubit reference is contextual, not the common-register encoding contrast.

In [ ]:
def single_qubit_gate(state,gate,q,n):
    shaped=state.reshape(-1,2**(n-q-1),2,2**q)
    a,b=shaped[:,:,0,:],shaped[:,:,1,:]
    return torch.stack((gate[0,0]*a+gate[0,1]*b,gate[1,0]*a+gate[1,1]*b),dim=2).reshape(-1,2**n)
def evolve_torch(state,angles,n,layers):
    for layer in range(layers):
        for q in range(n):
            rz,ry=angles[layer,q]
            zero=rz*0;one=zero+1
            zgate=torch.diag(torch.stack((torch.exp(-.5j*rz),torch.exp(.5j*rz))))
            c,s=torch.cos(ry/2),torch.sin(ry/2)
            ygate=torch.stack((torch.stack((c,-s)),torch.stack((s,c)))).to(torch.complex128)
            state=single_qubit_gate(state,zgate,q,n)
            state=single_qubit_gate(state,ygate,q,n)
        ix=torch.arange(2**n,device=state.device)
        for q in range(n):
            perm=ix^(((ix>>q)&1)<<(q+1 if q+1<n else 0))
            state=state[:,perm]
    return state
def branches(z,passive):
    if not passive:return z.to(torch.complex128),1
    n=z.shape[1]//2;left=z.clone();right=z.clone();left[:,n:]=0;right[:,:n]=0
    return torch.stack((left,right),dim=1).reshape(-1,z.shape[1]).to(torch.complex128),2
class CircuitHead(nn.Module):
    def __init__(self,qubits=5,passive=False,layers=2):
        super().__init__();self.qubits=qubits;self.passive=passive;self.layers=layers;self.dim=2**qubits
        self.angles=nn.Parameter(torch.randn(layers,qubits,2,dtype=torch.float64)*.3)
        self.readout=nn.Linear(qubits,K,dtype=torch.float64)
        signs=1-2*((np.arange(self.dim)[None,:]>>np.arange(qubits)[:,None])&1)
        self.register_buffer('signs',torch.tensor(signs,dtype=torch.float64))
    def unitary(self):
        return evolve_torch(torch.eye(self.dim,dtype=torch.complex128,device=self.angles.device),self.angles,self.qubits,self.layers).T
    def forward(self,z):
        state,b=branches(z,self.passive)
        state=evolve_torch(state,self.angles,self.qubits,self.layers)
        m=(state.abs().square()@self.signs.T).reshape(len(z),b,self.qubits).sum(1)
        return self.readout(m)
    def penalty(self):return self.readout.weight.square().sum()
class UnitaryHead(CircuitHead):
    """Full-unitary relaxation preserving shared observables and their spectra.

    A dense simulation control; it is NOT a shallow deployable quantum circuit.
    """
    def __init__(self,qubits=5,passive=False):
        super().__init__(qubits,passive,layers=1)
        del self.angles
        tri=torch.triu_indices(self.dim,self.dim)
        strict=torch.triu_indices(self.dim,self.dim,offset=1)
        self.register_buffer('tri',tri);self.register_buffer('strict',strict)
        self.h_real=nn.Parameter(torch.randn(tri.shape[1],dtype=torch.float64)*.02)
        self.h_imag=nn.Parameter(torch.randn(strict.shape[1],dtype=torch.float64)*.02)
    def unitary(self):
        r=torch.zeros((self.dim,self.dim),dtype=torch.float64,device=self.h_real.device).index_put(tuple(self.tri),self.h_real)
        s=torch.zeros_like(r).index_put(tuple(self.strict),self.h_imag)
        h=torch.complex(r+r.T-torch.diag(torch.diag(r)),s-s.T)
        return torch.matrix_exp(1j*h)
    def forward(self,z):
        state,b=branches(z,self.passive);state=state@self.unitary().T
        m=(state.abs().square()@self.signs.T).reshape(len(z),b,self.qubits).sum(1)
        return self.readout(m)
class LowRankHead(nn.Module):
    def __init__(self,dim=32,passive=False,rank=1):
        super().__init__();self.dim=dim;self.passive=passive
        self.v=nn.Parameter(torch.randn(dim,rank,dtype=torch.float64)/np.sqrt(dim))
        self.readout=nn.Linear(rank,K,dtype=torch.float64)
    def forward(self,z):
        state,b=branches(z,self.passive)
        a=(state.real@self.v).square().reshape(len(z),b,-1).sum(1)
        return self.readout(a)
    def penalty(self):return self.v.square().sum()+self.readout.weight.square().sum()
class TinyMLP(nn.Module):
    def __init__(self,input_dim=16,hidden=2):
        super().__init__();self.layers=nn.Sequential(nn.Linear(input_dim,hidden,dtype=torch.float64),nn.Tanh(),nn.Linear(hidden,K,dtype=torch.float64))
    def forward(self,x):return self.layers(x)
    def penalty(self):return sum(p.square().sum() for n,p in self.named_parameters() if n.endswith('weight'))
class ConvexSoftmax:
    def __init__(self,l2=.001):self.l2=l2
    def fit(self,x,y,weights,maxiter=1000):
        d=x.shape[1];targets=np.eye(K)[y];weights=weights/weights.sum()
        def objective(params):
            w=params[:d*K].reshape(d,K);b=params[d*K:];logits=x@w+b;logp=logits-logsumexp(logits,axis=1,keepdims=True)
            loss=-np.sum(weights[:,None]*targets*logp)+.5*self.l2*np.sum(w*w)
            dl=(np.exp(logp)-targets)*weights[:,None]
            return float(loss),np.r_[(x.T@dl+self.l2*w).ravel(),dl.sum(0)]
        start=time.perf_counter();res=minimize(objective,np.zeros(d*K+K),method='L-BFGS-B',jac=True,
            options={'maxiter':maxiter,'ftol':1e-10,'gtol':1e-7})
        self.w=res.x[:d*K].reshape(d,K);self.b=res.x[d*K:]
        self.info=dict(optimizer_success=bool(res.success),optimizer_message=str(res.message),iterations=int(res.nit),
            gradient_inf=float(np.abs(res.jac).max()),fit_seconds=time.perf_counter()-start,parameters=d*K+K)
        return self
    def logits(self,x):return x@self.w+self.b

## Cell 12 Numerical tests for information, unitarity, dephasing, equivalence and gradients

Check state information and unitarity, and verify $f_k(z)=z^\top A_kz+b_k$ for real amplitudes including the full learned readout. The passive matrices have zero cross blocks. The underlying complex observables preserve their spectra; taking their real part for real inputs need not preserve matrix eigenvalues. Compare autograd with central differences before training.

In [ ]:
def coefficient_matrices(net):
    with torch.no_grad():
        if isinstance(net,(CircuitHead,UnitaryHead)):
            u=net.unitary().cpu().numpy();diag=net.readout.weight.detach().cpu().numpy()@net.signs.cpu().numpy()
            a=np.stack([(u.conj().T@(d[:,None]*u)).real for d in diag])
            b=net.readout.bias.detach().cpu().numpy()
        elif isinstance(net,LowRankHead):
            v=net.v.detach().cpu().numpy();w=net.readout.weight.detach().cpu().numpy()
            a=np.einsum('ir,kr,jr->kij',v,w,v);b=net.readout.bias.detach().cpu().numpy()
        else:raise TypeError('This head has no quadratic state representation.')
        if net.passive:
            n=a.shape[1]//2;a[:,:n,n:]=0.;a[:,n:,:n]=0.
    return a,b
def forward_numpy(net,x):
    net.eval()
    with torch.no_grad():return net(torch.as_tensor(x,dtype=torch.float64,device=QDEVICE)).cpu().numpy()
rng=np.random.default_rng(10);test_x=rng.normal(size=(7,16));test_r0=np.median(np.linalg.norm(test_x,axis=1))
NUMERICAL_CHECKS=[]
for enc in ENCODINGS:
    z=encode_array(test_x,enc,test_r0);zm=encode_array(-test_x,enc,test_r0);passive=enc=='passive_norm'
    density_error=np.max(np.abs(encoded_density(z,passive)-encoded_density(zm,passive)))
    if enc in {'amplitude','passive_norm','coherent_norm'}:assert density_error<1e-12
    else:assert density_error>1e-5
    assert lift_array(z,passive).shape[1]==(272 if passive else 528)
    for kind,net in [('circuit',CircuitHead(passive=passive)),('unitary',UnitaryHead(passive=passive)),('lowrank',LowRankHead(passive=passive))]:
        net=net.to(QDEVICE)
        if hasattr(net,'unitary'):
            u=net.unitary().detach().cpu().numpy();np.testing.assert_allclose(u.conj().T@u,np.eye(32),atol=1e-10)
        a,b=coefficient_matrices(net);pred=forward_numpy(net,z)
        error=float(np.max(np.abs(pred-(np.einsum('ni,kij,nj->nk',z,a,z)+b))))
        assert error<1e-10
        NUMERICAL_CHECKS.append(dict(encoding=enc,head=kind,equivalence_error=error,sign_density_error=float(density_error)))
grad_model=CircuitHead().to(QDEVICE);zt=torch.tensor(encode_array(test_x,'signed_full',test_r0),dtype=torch.float64)
target=torch.tensor(np.arange(len(zt))%K)
loss=F.cross_entropy(grad_model(zt),target);loss.backward()
analytic=grad_model.angles.grad.detach().numpy().ravel().copy();finite=[];epsilon=1e-5
for j in range(min(6,len(analytic))):
    flat=grad_model.angles.view(-1)
    with torch.no_grad():flat[j]+=epsilon
    plus=F.cross_entropy(grad_model(zt),target).item()
    with torch.no_grad():flat[j]-=2*epsilon
    minus=F.cross_entropy(grad_model(zt),target).item()
    with torch.no_grad():flat[j]+=epsilon
    finite.append((plus-minus)/(2*epsilon))
gradient_error=float(np.max(np.abs(analytic[:len(finite)]-finite)))
assert gradient_error<1e-7
save_table('numerical_equivalence_checks.csv',NUMERICAL_CHECKS)
save_json('gradient_check.json',{'maximum_absolute_error':gradient_error,'passed':True})
print('Gradient check passed:',gradient_error,'| largest exact logit error:',max(r['equivalence_error'] for r in NUMERICAL_CHECKS))

## Cell 13 Synthetic sign, norm and direction information audit

Opposite-label synthetic pairs isolate information about global sign, magnitude or non-antipodal direction. A collision fraction of one means the prepared states are identical for those pairs. A noncollision does not imply easy learning, and a synthetic sign reversal is not a medically valid MRI counterfactual.

In [ ]:
rng=np.random.default_rng(112);u=rng.normal(size=(100,16));u/=np.linalg.norm(u,axis=1,keepdims=True)
v=rng.normal(size=u.shape);v-=(u*v).sum(1,keepdims=True)*u;v/=np.linalg.norm(v,axis=1,keepdims=True)
toy_rows=[]
for task,(a,b) in {'global_sign':(u,-u),'norm':(.5*u,2*u),'non_antipodal_direction':(u,v)}.items():
    for enc in ENCODINGS:
        za,zb=encode_array(a,enc,1.),encode_array(b,enc,1.)
        delta=np.max(np.abs(encoded_density(za,enc=='passive_norm')-encoded_density(zb,enc=='passive_norm')),axis=(1,2))
        toy_rows.append(dict(task=task,encoding=enc,opposite_label_pair_collision_fraction=float((delta<1e-12).mean())))
TOY_RESULTS=save_table('synthetic_information_audit.csv',toy_rows)
display(TOY_RESULTS.pivot(index='task',columns='encoding',values='opposite_label_pair_collision_fraction'))
print('These are engineered information-loss diagnostics, not MRI accuracy results.')

## Cell 14 Common fitting, prediction and validation-only hyperparameter selection

Neural heads use Adam, a fixed maximum epoch budget and validation checkpointing. The unrestricted lift uses convex multinomial cross-entropy with L2 regularization and L-BFGS. Regularization values have different functional effects across parameterizations; report each model's size, fitting budget and termination status. Convexity does not guarantee that a finite numerical run converged. SVM decision scores receive separate empirical temperature calibration.

In [ ]:
CANDIDATE_ROWS=[]; TRAINING_HISTORY=[]
def logits_of(net,features):
    return forward_numpy(net,features) if isinstance(net,nn.Module) else net.logits(features)
def fit_torch(net,xf,yf,uf,xv,yv,uv,l2,seed,tag):
    net=net.to(QDEVICE);weights=torch.tensor(unit_weights(uf),dtype=torch.float64,device=QDEVICE)
    tx=torch.tensor(xf,dtype=torch.float64,device=QDEVICE);ty=torch.tensor(yf,dtype=torch.long,device=QDEVICE)
    optimizer=torch.optim.Adam(net.parameters(),lr=CFG['lr']*(.5 if isinstance(net,UnitaryHead) else 1.))
    generator=torch.Generator().manual_seed(seed);best_key=(-np.inf,-np.inf);best_state=None;last_improve=0
    start=time.perf_counter();stopping='epoch_budget'
    for epoch in range(1,CFG['epochs']+1):
        net.train();order=torch.randperm(len(tx),generator=generator);epoch_loss=0.
        for batch in order.split(CFG['train_batch']):
            optimizer.zero_grad(set_to_none=True)
            prediction=net(tx[batch]);unregularized=F.cross_entropy(prediction,ty[batch],reduction='none')
            loss=(unregularized*weights[batch]).mean()+.5*l2*net.penalty()
            if not torch.isfinite(loss):raise FloatingPointError('Nonfinite training objective: '+tag)
            loss.backward();torch.nn.utils.clip_grad_norm_(net.parameters(),10.);optimizer.step()
            epoch_loss+=float(loss.detach())*len(batch)/len(tx)
        pv=softmax(logits_of(net,xv),axis=1);score=unit_metrics(yv,pv,uv);key=(score['macro_f1'],-score['nll'])
        TRAINING_HISTORY.append(dict(tag=tag,epoch=epoch,objective=epoch_loss,selection_macro_f1=score['macro_f1'],selection_nll=score['nll']))
        if key>best_key:
            best_key=key;best_state=copy.deepcopy(net.state_dict());last_improve=epoch
        if epoch-last_improve>=CFG['patience']:
            stopping='selection_early_stopping';break
    net.load_state_dict(best_state);net.eval()
    pf=softmax(logits_of(net,xf),axis=1)
    info=dict(parameters=sum(p.numel() for p in net.parameters()),fit_seconds=time.perf_counter()-start,
              epochs_completed=epoch,best_epoch=last_improve,termination=stopping,
              train_nll=unit_metrics(yf,pf,uf)['nll'],optimizer_success=None,
              optimizer_message='Adam with validation checkpointing; stationarity is not asserted.')
    return net,info
class SVMLogits:
    def __init__(self,c=1.):self.c=c
    def fit(self,x,y,w):
        start=time.perf_counter();self.svc=SVC(C=self.c,gamma='scale',decision_function_shape='ovr')
        self.svc.fit(x,y,sample_weight=w)
        self.info=dict(parameters=int(self.svc.support_vectors_.size+self.svc.dual_coef_.size+self.svc.intercept_.size),
            fit_seconds=time.perf_counter()-start,optimizer_success=bool(self.svc.fit_status_==0),
            optimizer_message='SVC decision scores are empirically temperature-calibrated, not native probabilities.')
        return self
    def logits(self,x):return self.svc.decision_function(x)
def fit_candidate(kind,encoding,seed,gamma,l2,prep,fit_idx,val_idx,tag):
    torch.manual_seed(seed);qubits=4 if kind=='circuit4' else 5
    ft=prep.transform(X_ALL[fit_idx]);fv=prep.transform(X_ALL[val_idx])
    xf=model_inputs(ft,encoding,prep.r0,gamma,kind,qubits);xv=model_inputs(fv,encoding,prep.r0,gamma,kind,qubits)
    yf,yv=Y_ALL[fit_idx],Y_ALL[val_idx];uf,uv=U_ALL[fit_idx],U_ALL[val_idx]
    passive=encoding=='passive_norm'
    if kind in {'circuit','circuit4'}:net=CircuitHead(qubits=qubits,passive=passive,layers=CFG['layers'])
    elif kind=='unitary':net=UnitaryHead(passive=passive)
    elif kind=='lowrank':net=LowRankHead(passive=passive)
    elif kind=='mlp':net=TinyMLP(xf.shape[1])
    elif kind in {'quadratic','linear_raw','linear_512'}:
        net=ConvexSoftmax(l2).fit(xf,yf,unit_weights(uf));info=net.info
    elif kind=='rbf':net=SVMLogits(l2).fit(xf,yf,unit_weights(uf));info=net.info
    else:raise ValueError('Unknown head '+kind)
    if isinstance(net,nn.Module):net,info=fit_torch(net,xf,yf,uf,xv,yv,uv,l2,seed,tag)
    score=unit_metrics(yv,softmax(logits_of(net,xv),axis=1),uv)
    rec=dict(id=tag,kind=kind,encoding=encoding,seed=int(seed),gamma=float(gamma),l2=float(l2),
             qubits=qubits,prep=prep,net=net,T=1.,info=info,selection=score)
    return rec
def tune(kind,encoding,seed,prep=PREP,fit_idx=None,val_idx=None,stage='main'):
    fit_idx=SPLITS['fit'] if fit_idx is None else fit_idx;val_idx=SPLITS['selection'] if val_idx is None else val_idx
    gammas=CFG['gamma_grid'] if encoding in {'signed_direction','signed_full'} else [1.]
    regularizers=[.1,1.,10.] if kind=='rbf' else CFG['l2_grid']
    best=None;key_best=(-np.inf,-np.inf,-np.inf)
    tag=f'{stage}__{encoding}__{kind}__s{seed}'
    for gamma in gammas:
        for reg in regularizers:
            candidate_tag=tag+f'__g{gamma}_r{reg}'
            rec=fit_candidate(kind,encoding,seed,gamma,reg,prep,fit_idx,val_idx,candidate_tag)
            CANDIDATE_ROWS.append(dict(stage=stage,kind=kind,encoding=encoding,seed=seed,gamma=gamma,l2=reg,
                                      **rec['selection'],**rec['info']))
            key=(rec['selection']['macro_f1'],-rec['selection']['nll'],reg if kind!='rbf' else -reg)
            if key>key_best:best,key_best=rec,key
    best['id']=tag
    return best
def record_logits(rec,raw_features):
    x=rec['prep'].transform(raw_features)
    inp=model_inputs(x,rec['encoding'],rec['prep'].r0,rec['gamma'],rec['kind'],rec['qubits'])
    return logits_of(rec['net'],inp)
def record_probability(rec,raw_features,calibrated=True):
    return softmax(record_logits(rec,raw_features)/(rec['T'] if calibrated else 1.),axis=1)
def record_summary(rec):
    return dict(id=rec['id'],kind=rec['kind'],encoding=rec['encoding'],seed=rec['seed'],gamma=rec['gamma'],
                l2=rec['l2'],qubits=rec['qubits'],temperature=rec['T'],**rec['info'])

## Cell 15 Train the controlled encoding grid and classical reference models

Train each requested reference encoding with shared circuit, quadratic and low-rank heads. Add full-unitary controls for coherent norm and signed full. Repeat stochastic heads across all configured seeds. Convex heads are deterministic on the fixed split and are fitted once. Raw PCA16 MLP/logistic/RBF models are representation controls; the 512-feature logistic probe is a richer-input practical comparator, not a matched quantum-state comparator.

In [ ]:
RECORDS=[]
for encoding in ENCODINGS:
    for kind in ['circuit','quadratic','lowrank']+(['unitary'] if CFG['run_unitary_control'] and encoding in {'coherent_norm','signed_full'} else []):
        seeds=[CFG['seeds'][0]] if kind=='quadratic' else CFG['seeds']
        for seed in seeds:
            rec=tune(kind,encoding,seed);RECORDS.append(rec)
            print('Selected',rec['id'],'gamma',rec['gamma'],'L2',rec['l2'],'validation F1',round(rec['selection']['macro_f1'],3),flush=True)
for seed in CFG['seeds']:
    RECORDS.append(tune('circuit4','amplitude',seed))
    RECORDS.append(tune('mlp','raw_pca16',seed))
RECORDS.append(tune('linear_raw','raw_pca16',CFG['seeds'][0]))
RECORDS.append(tune('rbf','raw_pca16',CFG['seeds'][0]))
PREP512=FeaturePrep(None).fit(X_ALL[SPLITS['fit']])
RECORDS.append(tune('linear_512','frozen_resnet512',CFG['seeds'][0],prep=PREP512))
save_table('candidate_selection_results.csv',CANDIDATE_ROWS)
save_table('training_history.csv',TRAINING_HISTORY)
display(pd.DataFrame([record_summary(r) for r in RECORDS])[['kind','encoding','seed','parameters','fit_seconds']])

## Cell 16 Nested patient or proxy-group development validation

Outer folds measure development refitting variation. Each outer training partition gets an inner patient/group-disjoint selection holdout. Scaler, PCA, model fitting and hyperparameter selection occur only inside that outer training partition. These development folds do not create independent external patients, and overlapping folds are not independent observations.

In [ ]:
CV_ROWS=[];CV_PREDICTIONS=[]
if CFG['run_grouped_cv']:
    from sklearn.model_selection import StratifiedKFold
    dev_idx=np.concatenate([SPLITS['fit'],SPLITS['selection'],SPLITS['temperature']])
    group_tab=manifest.iloc[dev_idx][['split_group','y']].drop_duplicates().reset_index(drop=True)
    skf=StratifiedKFold(CFG['cv_folds'],shuffle=True,random_state=CFG['split_seed']+20)
    cv_specs=[('circuit','amplitude'),('circuit','signed_direction'),('circuit','signed_full'),('quadratic','coherent_norm')]
    for fold,(tr,ev) in enumerate(skf.split(group_tab.split_group,group_tab.y)):
        outer_train=group_tab.iloc[tr];outer_eval=group_tab.iloc[ev]
        inner_fit,inner_val=train_test_split(outer_train.split_group,test_size=.25,stratify=outer_train.y,
                                            random_state=CFG['split_seed']+fold)
        fi=dev_idx[manifest.iloc[dev_idx].split_group.isin(inner_fit).to_numpy()]
        vi=dev_idx[manifest.iloc[dev_idx].split_group.isin(inner_val).to_numpy()]
        ei=dev_idx[manifest.iloc[dev_idx].split_group.isin(outer_eval.split_group).to_numpy()]
        if len(fi)<17:
            CV_ROWS.append(dict(fold=fold,status='skipped_insufficient_rows_for_PCA16'));continue
        fold_prep=FeaturePrep(16).fit(X_ALL[fi])
        for kind,encoding in cv_specs:
            seed=CFG['seeds'][0]
            rec=tune(kind,encoding,seed,fold_prep,fi,vi,stage=f'cv_fold{fold}')
            p=record_probability(rec,X_ALL[ei],calibrated=False)
            CV_ROWS.append(dict(fold=fold,kind=kind,encoding=encoding,status='complete',
                                **unit_metrics(Y_ALL[ei],p,U_ALL[ei])))
            ids,yy,pp=aggregate(Y_ALL[ei],p,U_ALL[ei])
            for unit,label,prob in zip(ids,yy,pp):
                CV_PREDICTIONS.append(dict(fold=fold,kind=kind,encoding=encoding,analysis_unit=unit,y=int(label),
                                           **{f'p{j}':float(v) for j,v in enumerate(prob)}))
save_table('nested_grouped_development_cv.csv',CV_ROWS);save_table('nested_grouped_cv_predictions.csv',CV_PREDICTIONS)
display(pd.DataFrame(CV_ROWS))
print('Outer folds use disjoint evaluation groups; scaler/PCA and hyperparameter selection are refitted inside each outer training partition.')

## Cell 17 Realizable teachers with Adam and L-BFGS optimization

A teacher of the same architecture supplies probabilities. Compare multiple teachers/seeds with Adam and L-BFGS. Training KL zero is attainable by the teacher parameters. Failure under a stated budget supports an optimization limitation; success does not establish MRI expressivity. Iteration counts across optimizers are not identical computational budgets, so record elapsed time.

In [ ]:
TEACHER_RESULTS=[]
if CFG['run_teacher']:
    for enc in ['coherent_norm','signed_full']:
        for teacher_seed in CFG['teacher_seeds']:
            rng=np.random.default_rng(teacher_seed);x=rng.normal(size=(96,16));r0=np.median(np.linalg.norm(x[:64],axis=1))
            z=torch.tensor(encode_array(x,enc,r0),dtype=torch.float64,device=QDEVICE)
            torch.manual_seed(teacher_seed);teacher=CircuitHead().to(QDEVICE)
            with torch.no_grad():
                teacher.angles.normal_(0,1);teacher.readout.weight.normal_(0,1.5)
                targets=F.softmax(teacher(z),dim=1)
            teacher_target_gap=float(F.kl_div(F.log_softmax(teacher(z[:64]),dim=1),targets[:64],reduction='batchmean').detach())
            assert abs(teacher_target_gap)<1e-10
            for seed in CFG['seeds']:
                for optimizer_name in ['Adam','LBFGS']:
                    torch.manual_seed(seed);student=CircuitHead().to(QDEVICE);start=time.perf_counter()
                    if optimizer_name=='Adam':
                        opt=torch.optim.Adam(student.parameters(),lr=.02)
                        for step in range(CFG['teacher_steps']):
                            opt.zero_grad();loss=F.kl_div(F.log_softmax(student(z[:64]),dim=1),targets[:64],reduction='batchmean')
                            loss.backward();opt.step()
                    else:
                        opt=torch.optim.LBFGS(student.parameters(),max_iter=CFG['teacher_steps'],line_search_fn='strong_wolfe',tolerance_grad=1e-8)
                        def closure():
                            opt.zero_grad();loss=F.kl_div(F.log_softmax(student(z[:64]),dim=1),targets[:64],reduction='batchmean')
                            loss.backward();return loss
                        opt.step(closure)
                    with torch.no_grad():
                        tr=float(F.kl_div(F.log_softmax(student(z[:64]),dim=1),targets[:64],reduction='batchmean'))
                        ev=float(F.kl_div(F.log_softmax(student(z[64:]),dim=1),targets[64:],reduction='batchmean'))
                    TEACHER_RESULTS.append(dict(encoding=enc,teacher_seed=teacher_seed,student_seed=seed,optimizer=optimizer_name,
                        train_kl=max(0.,tr),heldout_kl=max(0.,ev),teacher_attainable_kl=teacher_target_gap,
                        optimization_budget=CFG['teacher_steps'],seconds=time.perf_counter()-start))
save_table('realizable_teacher_recovery.csv',TEACHER_RESULTS)
display(pd.DataFrame(TEACHER_RESULTS))

## Cell 18 Train learning curves before opening evaluation predictions

Use nested class-stratified group subsets at 10%, 25%, 50% and 100%. Refit scaler/PCA for each subset. Compare the frozen CNN + quantum head with its low-rank, unrestricted quadratic and compact MLP controls. These are fixed-hyperparameter learning curves; choices came from development data. Tiny subsets unable to fit PCA16 are explicitly skipped. This is not a reproduction of the original report's different end-to-end CNN.

In [ ]:
CURVE_RECORDS=[];CURVE_STATUS=[]
if CFG['run_learning_curves']:
    fit_tab=manifest.iloc[SPLITS['fit']][['split_group','y']].drop_duplicates()
    for seed in CFG['seeds']:
        order={}
        rng=np.random.default_rng(seed+1000)
        for label,part in fit_tab.groupby('y'):order[label]=rng.permutation(sorted(part.split_group))
        for fraction in CFG['learning_fractions']:
            chosen=np.concatenate([ids[:max(1,math.ceil(len(ids)*fraction))] for ids in order.values()])
            fi=SPLITS['fit'][manifest.iloc[SPLITS['fit']].split_group.isin(chosen).to_numpy()]
            if len(fi)<17:
                CURVE_STATUS.append(dict(seed=seed,fraction=fraction,status='skipped: fewer than 17 training rows for PCA16'));continue
            prep=FeaturePrep(16).fit(X_ALL[fi])
            for kind,enc in [('circuit','signed_full'),('quadratic','signed_full'),('lowrank','signed_full'),('mlp','raw_pca16')]:
                # Fixed-hyperparameter curves: settings were selected on development
                # data; each fraction refits its own scaler/PCA and model.
                template=next(r for r in RECORDS if r['kind']==kind and r['encoding']==enc and
                              (r['seed']==seed or kind=='quadratic'))
                rec=fit_candidate(kind,enc,seed,template['gamma'],template['l2'],prep,fi,SPLITS['selection'],
                                  f'curve_f{fraction}__{enc}__{kind}__s{seed}')
                rec['fraction']=fraction;rec['n_training_units']=manifest.iloc[fi].analysis_unit.nunique()
                CURVE_RECORDS.append(rec)
                CURVE_STATUS.append(dict(seed=seed,fraction=fraction,kind=kind,status='trained',n_training_units=rec['n_training_units']))
save_table('learning_curve_training_status.csv',CURVE_STATUS)
display(pd.DataFrame(CURVE_STATUS))

## Cell 19 Fit a border-and-dimension shortcut probe on development data

A classifier sees only original dimensions and outer-border statistics. No filenames or folder names are used. Its validation result is a shortcut screen; border pixels can contain anatomical edges, so success alone does not prove confounding.

In [ ]:
def border_features(frame):
    output=[]
    for row in frame.itertuples(index=False):
        im=read_gray(row.path);h,w=im.shape;features=[math.log(h),math.log(w),h/w]
        for fraction in [.05,.10]:
            border=np.ones_like(im,dtype=bool);bh=max(1,int(h*fraction));bw=max(1,int(w*fraction))
            border[bh:-bh,bw:-bw]=False;vals=im[border]
            features.extend([vals.mean(),vals.std(),*np.quantile(vals,[.1,.5,.9]),float((vals==0).mean())])
        output.append(features)
    return np.asarray(output)
BORDER_X=border_features(manifest);border_scaler=StandardScaler().fit(BORDER_X[SPLITS['fit']])
best_border=None;best_key=(-np.inf,-np.inf)
for reg in CFG['l2_grid']:
    net=ConvexSoftmax(reg).fit(border_scaler.transform(BORDER_X[SPLITS['fit']]),Y_ALL[SPLITS['fit']],unit_weights(U_ALL[SPLITS['fit']]))
    p=softmax(net.logits(border_scaler.transform(BORDER_X[SPLITS['selection']])),axis=1)
    score=unit_metrics(Y_ALL[SPLITS['selection']],p,U_ALL[SPLITS['selection']]);key=(score['macro_f1'],-score['nll'])
    if key>best_key:best_border,best_key=net,key
print('Shortcut probe selection F1:',best_key[0],'. Border pixels may include anatomical edges; this probe alone does not establish confounding.')

## Cell 20 Freeze all choices and analysis definitions before evaluation

Serialize selected models, primary contrasts and the analysis protocol, then compute a SHA256 record. This makes the run reviewable but does not constitute prospective preregistration or make previously inspected test patients unseen.

In [ ]:
frozen=dict(config=CFG,classes=CLASSES,unit_kind=UNIT_KIND,
    primary_prediction='Mean of temperature-calibrated slice probabilities within each analysis unit; compare average macro-F1 across prespecified fitted seeds.',
    selected_models=[record_summary(r) for r in RECORDS],
    primary_contrasts=[['signed_direction_circuit','amplitude_circuit'],
                       ['signed_full_circuit','signed_direction_circuit'],
                       ['coherent_norm_circuit','coherent_norm_quadratic']],
    inference='Paired analysis-unit bootstrap/permutation; same patient/group resamples across all fitted seeds; Holm across 3 primary contrasts.',
    interval_scope='Conditional on fitted models; refitting variability described by nested grouped development validation.',
    limits=['No inferred patient identifiers','No fabricated tumor masks','No inference of independent provenance from folder names',
            'Reference circuits are explicitly defined here; the original unpublished circuit equations were not supplied.'])
protocol_text=json.dumps(frozen,sort_keys=True,default=str)
PROTOCOL_SHA=hashlib.sha256(protocol_text.encode()).hexdigest()
save_json('frozen_protocol.json',frozen);(RUN_DIR/'protocol_sha256.txt').write_text(PROTOCOL_SHA+'\n')
save_table('candidate_selection_results.csv',CANDIDATE_ROWS);save_table('training_history.csv',TRAINING_HISTORY)
print('Frozen protocol SHA256:',PROTOCOL_SHA)
print('No evaluation predictions have been used to select these models.')

## Cell 21 Independent temperature calibration and checkpoint export

Fit one positive temperature per selected model using only the reserved temperature partition. Save preprocessing, model state, raw quadratic equivalents and calibration constants. Small class-specific calibration counts are reported. No conformal guarantees are claimed.

In [ ]:
def fit_temperature(logits,y,units):
    def objective(logt):
        _,yy,pp=aggregate(y,softmax(logits/np.exp(logt),axis=1),units)
        return nll(yy,pp)
    result=minimize_scalar(objective,bounds=(math.log(.05),math.log(20.)),method='bounded')
    return float(np.exp(result.x))
def calibrate_record(rec):
    ci=SPLITS['temperature'];logits=record_logits(rec,X_ALL[ci])
    rec['T']=fit_temperature(logits,Y_ALL[ci],U_ALL[ci]);return rec
def save_record(rec):
    name=rec['id'];net=rec['net'];rec['prep'].save(RUN_DIR/'checkpoints'/(name+'__prep.npz'))
    if isinstance(net,nn.Module):
        torch.save(net.state_dict(),RUN_DIR/'checkpoints'/(name+'.pt'))
        if isinstance(net,(CircuitHead,UnitaryHead,LowRankHead)):
            a,b=coefficient_matrices(net)
            np.savez_compressed(RUN_DIR/'checkpoints'/(name+'__quadratic_equivalent.npz'),A=a,b=b,T=rec['T'])
    elif isinstance(net,ConvexSoftmax):
        np.savez_compressed(RUN_DIR/'checkpoints'/(name+'.npz'),w=net.w,b=net.b,T=rec['T'])
    elif isinstance(net,SVMLogits):
        import joblib
        joblib.dump(net.svc,RUN_DIR/'checkpoints'/(name+'.joblib'))
    save_json('checkpoints/'+name+'__metadata.json',record_summary(rec))
for rec in RECORDS+CURVE_RECORDS:
    calibrate_record(rec);save_record(rec)
calibration_counts=manifest.iloc[SPLITS['temperature']][['analysis_unit','label']].drop_duplicates().groupby('label').size().to_dict()
save_json('calibration_counts.json',calibration_counts)
print('Calibration units per class:',calibration_counts)
if min(calibration_counts.values())<20:
    print('Calibration sample is small; temperature estimates and ECE are exploratory. No conformal coverage guarantee is claimed.')
save_table('selected_model_configuration.csv',[record_summary(r) for r in RECORDS])

## Cell 22 Evaluate locked models and export prediction-level results

Export raw/calibrated metrics and complete unit-level class probabilities. Recheck exact quadratic equality on actual evaluation images. The primary prediction is the mean of calibrated slice probabilities; primary effects average macro-F1 across the prespecified fitted seeds.

In [ ]:
ei=SPLITS['evaluation'];ey=Y_ALL[ei];eu=U_ALL[ei]
EVAL_UNIT_IDS,EVAL_UNIT_Y,_=aggregate(ey,np.ones((len(ei),K))/K,eu)
EVAL_PROBS={};EVAL_SLICE_PROBS={};EVAL_METRICS=[];PREDICTION_ROWS=[];PARITY_ROWS=[]
for rec in RECORDS:
    logits=record_logits(rec,X_ALL[ei]);pr=softmax(logits,axis=1);pc=softmax(logits/rec['T'],axis=1)
    EVAL_SLICE_PROBS[rec['id']]=pc
    ids,yy,pp=aggregate(ey,pc,eu);assert np.array_equal(ids,EVAL_UNIT_IDS)
    EVAL_PROBS[rec['id']]=pp
    for calibrated,p in [(False,pr),(True,pc)]:
        EVAL_METRICS.append(dict(model=rec['id'],kind=rec['kind'],encoding=rec['encoding'],seed=rec['seed'],
                                  calibrated=calibrated,unit_kind=UNIT_KIND,**unit_metrics(ey,p,eu)))
    for unit,label,p in zip(ids,yy,pp):
        PREDICTION_ROWS.append(dict(model=rec['id'],analysis_unit=unit,y=int(label),
                                   **{f'p{j}':float(v) for j,v in enumerate(p)}))
    if isinstance(rec['net'],(CircuitHead,UnitaryHead,LowRankHead)):
        x=rec['prep'].transform(X_ALL[ei]);z=encode_array(x,rec['encoding'],rec['prep'].r0,rec['gamma'],rec['qubits'])
        a,b=coefficient_matrices(rec['net']);surrogate=np.einsum('ni,kij,nj->nk',z,a,z)+b
        error=float(np.abs(logits-surrogate).max());assert error<1e-9
        PARITY_ROWS.append(dict(model=rec['id'],maximum_logit_error=error))
METRICS_FRAME=save_table('evaluation_metrics.csv',EVAL_METRICS)
save_table('evaluation_unit_predictions.csv',PREDICTION_ROWS);save_table('real_image_exact_equivalence.csv',PARITY_ROWS)
# Shortcut probe uses the same separate temperature partition.
ci=SPLITS['temperature'];btemp=fit_temperature(best_border.logits(border_scaler.transform(BORDER_X[ci])),Y_ALL[ci],U_ALL[ci])
border_p=softmax(best_border.logits(border_scaler.transform(BORDER_X[ei]))/btemp,axis=1)
save_table('border_dimension_probe.csv',[dict(temperature=btemp,unit_kind=UNIT_KIND,**unit_metrics(ey,border_p,eu))])
display(METRICS_FRAME[METRICS_FRAME.calibrated].groupby(['kind','encoding']).macro_f1.agg(['mean','std','count']).reset_index())

## Cell 23 Learning curve evaluation and resource accounting

Evaluate every already trained curve point using the same evaluation units. Report training-unit counts, fit time and parameters. Circuit gate counts cover only the trainable ansatz; data preparation, routing and measurement shots were not synthesized. CPU simulator time is not quantum hardware time or evidence of speedup.

In [ ]:
CURVE_RESULTS=[]
for rec in CURVE_RECORDS:
    p=record_probability(rec,X_ALL[ei])
    CURVE_RESULTS.append(dict(model=rec['id'],kind=rec['kind'],encoding=rec['encoding'],seed=rec['seed'],
        fraction=rec['fraction'],n_training_units=rec['n_training_units'],fit_seconds=rec['info']['fit_seconds'],
        parameters=rec['info']['parameters'],**unit_metrics(ey,p,eu)))
CURVE_FRAME=save_table('learning_curve_evaluation.csv',CURVE_RESULTS)
resource=[]
for rec in RECORDS:
    quantum=rec['kind'] in {'circuit','circuit4'};q=rec['qubits']
    resource.append(dict(model=rec['id'],kind=rec['kind'],parameters=rec['info']['parameters'],
        fit_seconds=rec['info']['fit_seconds'],ansatz_qubits=q if quantum else np.nan,
        ansatz_rotation_gates=2*q*CFG['layers'] if quantum else np.nan,
        ansatz_cnot_gates=q*CFG['layers'] if quantum else np.nan,
        state_preparation_synthesized=False,hardware_executed=False,
        note='CPU simulation fit time; ansatz-only gates exclude data loading, routing and shots.'))
RESOURCE_FRAME=save_table('resource_accounting.csv',resource)

## Cell 24 Paired bootstrap and permutation tests for three prespecified contrasts

Primary contrasts: signed direction minus amplitude; signed full minus signed direction; coherent circuit minus its exact-lift quadratic comparator. Resample whole analysis units, stratified by class; use the same sampled units across all fitted seeds. Permutation swaps both models' predictions together across seeds for each unit. Holm correction covers the three two-sided permutation tests. Confidence intervals are pointwise and conditional on fitted models; they exclude refitting uncertainty. Proxy-group inference remains exploratory.

In [ ]:
def fitted_stack(encoding,kind):
    recs=sorted([r for r in RECORDS if r['encoding']==encoding and r['kind']==kind],key=lambda r:r['seed'])
    if not recs:raise ValueError('Missing prespecified comparator.')
    return np.stack([EVAL_PROBS[r['id']] for r in recs])
def align_stacks(a,b):
    if len(a)==len(b):return a,b
    if len(a)==1:return np.repeat(a,len(b),axis=0),b
    if len(b)==1:return a,np.repeat(b,len(a),axis=0)
    raise ValueError('Unmatched stochastic seed sets.')
def paired_inference(y,a,b,bootstrap=None,permutations=None,seed=18,unit_kind=None):
    a,b=align_stacks(a,b);pa=a.argmax(2);pb=b.argmax(2)
    def delta(indices,aa=pa,bb=pb):
        return float(np.mean([macro_f1_fast(y[indices],u[indices])-macro_f1_fast(y[indices],v[indices]) for u,v in zip(aa,bb)]))
    all_idx=np.arange(len(y));observed=delta(all_idx);rng=np.random.default_rng(seed)
    strata=[np.flatnonzero(y==j) for j in range(K) if np.any(y==j)]
    nb=CFG['bootstrap'] if bootstrap is None else bootstrap;np_=CFG['permutation'] if permutations is None else permutations
    boot=np.array([delta(np.concatenate([rng.choice(s,len(s),replace=True) for s in strata])) for _ in range(nb)])
    null=[]
    for _ in range(np_):
        swap=rng.random(len(y))<.5
        aa=np.where(swap[None,:],pb,pa);bb=np.where(swap[None,:],pa,pb)
        null.append(delta(all_idx,aa,bb))
    p=(1+np.sum(np.abs(null)>=abs(observed)-1e-14))/(1+np_)
    low,high=np.quantile(boot,[.025,.975])
    return dict(delta_macro_f1=observed,ci_low=float(low),ci_high=float(high),permutation_p_two_sided=float(p),
        excludes_improvement_over_margin=bool(high<CFG['equivalence_margin']),
        within_equivalence_margin=bool(low>-CFG['equivalence_margin'] and high<CFG['equivalence_margin']),
        n_analysis_units=len(y),n_fitted_seed_pairs=len(a),unit_kind=UNIT_KIND if unit_kind is None else unit_kind,
        interval_scope='pointwise 95%; conditional on these fitted models; no refit variance',
        exploratory_patient_independence_unverified=(UNIT_KIND if unit_kind is None else unit_kind)!='verified_patient')
contrasts=[('signed_direction_minus_amplitude',('signed_direction','circuit'),('amplitude','circuit')),
           ('signed_full_minus_signed_direction',('signed_full','circuit'),('signed_direction','circuit')),
           ('coherent_minus_same_lift_quadratic',('coherent_norm','circuit'),('coherent_norm','quadratic'))]
PRIMARY_RESULTS=[]
for j,(name,a,b) in enumerate(contrasts):
    PRIMARY_RESULTS.append(dict(contrast=name,**paired_inference(EVAL_UNIT_Y,fitted_stack(*a),fitted_stack(*b),seed=100+j)))
order=np.argsort([r['permutation_p_two_sided'] for r in PRIMARY_RESULTS]);previous=0.
for rank,j in enumerate(order):
    adjusted=min(1.,max(previous,(len(order)-rank)*PRIMARY_RESULTS[j]['permutation_p_two_sided']))
    PRIMARY_RESULTS[j]['holm_adjusted_p']=adjusted;previous=adjusted
PRIMARY_FRAME=save_table('primary_paired_inference.csv',PRIMARY_RESULTS)
identity=paired_inference(EVAL_UNIT_Y,fitted_stack('amplitude','circuit'),fitted_stack('amplitude','circuit'),bootstrap=20,permutations=19)
assert identity['delta_macro_f1']==0 and identity['ci_low']==0 and identity['ci_high']==0
display(PRIMARY_FRAME[['contrast','delta_macro_f1','ci_low','ci_high','holm_adjusted_p','unit_kind']])

## Cell 25 Figures for accuracy, uncertainty, learning curves and confusion matrices

Produce exportable PNGs. Error bars across seeds are standard deviations, not patient confidence intervals. Confusion matrices show fixed, declared models and seed zero rather than a test-selected winner.

In [ ]:
cal=METRICS_FRAME[METRICS_FRAME.calibrated]
subset=cal[cal.kind.isin(['circuit','quadratic','lowrank','unitary'])]
summary=subset.groupby(['encoding','kind']).macro_f1.agg(['mean','std']).reset_index()
fig,ax=plt.subplots(figsize=(11,5))
labels=[];means=[];errors=[]
for row in summary.itertuples():
    labels.append(row.encoding+' / '+row.kind);means.append(row.mean);errors.append(0. if pd.isna(row.std) else row.std)
ax.bar(np.arange(len(means)),means,yerr=errors,capsize=2);ax.set_xticks(np.arange(len(means)),labels,rotation=70,ha='right')
ax.set_ylabel('Unit macro-F1; error bars are seed SD');ax.set_ylim(0,1);fig.tight_layout()
fig.savefig(RUN_DIR/'figures'/'encoding_comparison.png',dpi=180);plt.show();plt.close(fig)
fig,axes=plt.subplots(1,3,figsize=(12,4))
for ax,(enc,kind) in zip(axes,[('amplitude','circuit'),('coherent_norm','circuit'),('signed_full','circuit')]):
    rec=next(r for r in RECORDS if r['encoding']==enc and r['kind']==kind and r['seed']==CFG['seeds'][0])
    cm=confusion_matrix(EVAL_UNIT_Y,EVAL_PROBS[rec['id']].argmax(1),labels=np.arange(K))
    ax.imshow(cm,cmap='Blues');ax.set_xticks(range(K),CLASSES,rotation=60,ha='right');ax.set_yticks(range(K),CLASSES)
    for a in range(K):
        for b in range(K):ax.text(b,a,str(cm[a,b]),ha='center',va='center')
    ax.set_title(enc);ax.set_xlabel('Predicted');ax.set_ylabel('Observed')
fig.tight_layout();fig.savefig(RUN_DIR/'figures'/'confusion_matrices.png',dpi=180);plt.show();plt.close(fig)
if not CURVE_FRAME.empty:
    fig,ax=plt.subplots(figsize=(7,4))
    for kind,part in CURVE_FRAME.groupby('kind'):
        g=part.groupby('fraction').agg(n=('n_training_units','mean'),score=('macro_f1','mean'),sd=('macro_f1','std')).sort_index()
        ax.errorbar(g.n,g.score,yerr=g.sd.fillna(0),marker='o',capsize=3,label=kind)
    ax.set_xlabel('Training analysis units');ax.set_ylabel('Evaluation macro-F1; seed SD');ax.legend();fig.tight_layout()
    fig.savefig(RUN_DIR/'figures'/'learning_curves.png',dpi=180);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(6,4))
for enc in ['coherent_norm','signed_full']:
    rec=next(r for r in RECORDS if r['encoding']==enc and r['kind']=='circuit' and r['seed']==CFG['seeds'][0])
    pp=EVAL_PROBS[rec['id']];conf=pp.max(1);correct=pp.argmax(1)==EVAL_UNIT_Y
    xx=[];yy=[]
    for lo in np.linspace(0,.9,10):
        mask=(conf>=lo)&(conf<lo+.1 if lo<.9 else conf<=1)
        if mask.any():xx.append(conf[mask].mean());yy.append(correct[mask].mean())
    ax.plot(xx,yy,'o-',label=enc)
ax.plot([0,1],[0,1],'k--');ax.set(xlim=(0,1),ylim=(0,1),xlabel='Mean confidence',ylabel='Accuracy');ax.legend();fig.tight_layout()
fig.savefig(RUN_DIR/'figures'/'reliability.png',dpi=180);plt.show();plt.close(fig)

## Cell 26 Feature-level norm interventions with all fitted constants frozen

Multiply the PCA16 vector by positive constants while keeping direction fixed. Keep training-derived scales and the fitted model frozen. Verify exact scale invariance for direction-only physical encoders. This is a feature-space mechanism test, not an MRI scanner simulation.

In [ ]:
FEATURE_STRESS_ROWS=[];FEATURE_STRESS_PROBS={}
for rec in RECORDS:
    if rec['kind']=='linear_512':continue # Different input dimensionality; keep it out of PCA16 mechanism tests.
    x0=rec['prep'].transform(X_ALL[ei]);baseline=EVAL_PROBS[rec['id']]
    for scale in CFG['norm_scales']:
        inputs=model_inputs(scale*x0,rec['encoding'],rec['prep'].r0,rec['gamma'],rec['kind'],rec['qubits'])
        p=softmax(logits_of(rec['net'],inputs)/rec['T'],axis=1);ids,yy,pp=aggregate(ey,p,eu)
        FEATURE_STRESS_PROBS[(rec['id'],scale)]=pp
        FEATURE_STRESS_ROWS.append(dict(model=rec['id'],kind=rec['kind'],encoding=rec['encoding'],seed=rec['seed'],scale=scale,
            mean_unit_total_variation=float(np.mean(np.abs(pp-baseline).sum(1)/2)),
            unit_prediction_flip_rate=float(np.mean(pp.argmax(1)!=baseline.argmax(1))),**unit_metrics(ey,p,eu)))
        if rec['encoding'] in {'amplitude','signed_direction'} and rec['kind'] in {'circuit','circuit4','lowrank','quadratic','unitary'}:
            assert np.max(np.abs(pp-baseline))<1e-9,'Direction-only encoder failed the positive-scale invariance test.'
FEATURE_STRESS_FRAME=save_table('feature_norm_stress.csv',FEATURE_STRESS_ROWS)
fig,ax=plt.subplots(figsize=(7,4))
for enc,part in FEATURE_STRESS_FRAME[FEATURE_STRESS_FRAME.kind=='circuit'].groupby('encoding'):
    g=part.groupby('scale').macro_f1.agg(['mean','std']);ax.errorbar(g.index,g['mean'],yerr=g['std'].fillna(0),marker='o',label=enc)
ax.set(xlabel='PCA-vector norm multiplier',ylabel='Evaluation unit macro-F1');ax.legend(fontsize=8);fig.tight_layout()
fig.savefig(RUN_DIR/'figures'/'feature_norm_sensitivity.png',dpi=180);plt.show();plt.close(fig)

## Cell 27 Image-level gain, offset, smooth bias and exact-zero border stress tests

Change raw images, then rerun the complete original normalization, resize, frozen backbone, fitted scaler/PCA, encoding and model. Compare all models on the same prespecified stress subset; use all eligible evaluation units in full mode. Exact-zero border changes preserve nonzero source pixels but are not validated anatomical masks. Strong robustness after robust normalization can arise from preprocessing.

In [ ]:
def stress_subset(frame,per_class):
    if per_class is None:return frame.copy()
    tab=frame[['analysis_unit','y']].drop_duplicates();rng=np.random.default_rng(CFG['split_seed']+300);chosen=[]
    for label,part in tab.groupby('y'):
        chosen.extend(rng.choice(sorted(part.analysis_unit),min(per_class,len(part)),replace=False).tolist())
    return frame[frame.analysis_unit.isin(chosen)].copy()
def perturb_image(im,row,kind,strength):
    if kind=='gain':return im*strength
    if kind=='offset':return im+strength
    seed=int(hashlib.sha256((row.pixel_sha256+'|'+kind).encode()).hexdigest()[:8],16)
    rng=np.random.default_rng(seed)
    if kind=='bias':
        h,w=im.shape;yy,xx=np.meshgrid(np.linspace(-1,1,h),np.linspace(-1,1,w),indexing='ij')
        c=rng.normal(size=5);field=c[0]*xx+c[1]*yy+c[2]*xx*yy+c[3]*xx**2+c[4]*yy**2
        field=(field-field.mean())/max(field.std(),1e-8)
        return im*np.exp(strength*field)
    if kind=='zero_border':
        h,w=im.shape;bh=max(1,int(h*.04));bw=max(1,int(w*.04))
        border=np.ones_like(im,dtype=bool);border[bh:-bh,bw:-bw]=False
        editable=border&(im==0);out=im.copy();out[editable]=rng.uniform(0,strength,editable.sum())
        assert np.array_equal(out[im>0],im[im>0])
        return out
    raise ValueError('Unknown image intervention.')
IMAGE_STRESS_ROWS=[];IMAGE_STRESS_PROBS={};IMAGE_STRESS_METADATA=[]
stress_frame=stress_subset(manifest.iloc[ei],CFG['stress_units_per_class'])
stress_idx=stress_frame.index.to_numpy();sx=X_ALL[stress_idx];sy=Y_ALL[stress_idx];su=U_ALL[stress_idx]
stress_ids,stress_y,_=aggregate(sy,np.ones((len(sy),K))/K,su)
save_table('image_stress_manifest.csv',stress_frame)
IMAGE_BASE_PROBS={}
for rec in RECORDS:
    _,_,IMAGE_BASE_PROBS[rec['id']]=aggregate(sy,record_probability(rec,sx),su)
if CFG['run_image_stress']:
    conditions=[('gain',.75),('gain',1.25),('offset',-.10),('offset',.10),('bias',.15),('bias',.30),('zero_border',.20)]
    for kind,strength in conditions:
        changed=[]
        for row in stress_frame.itertuples(index=False):
            raw=read_gray(row.path);altered=perturb_image(raw,row,kind,strength)
            changed.append(float((np.abs(altered-raw)>1e-12).mean()))
        IMAGE_STRESS_METADATA.append(dict(condition=kind,strength=strength,images=len(changed),
            images_with_raw_pixel_changes=int(np.sum(np.asarray(changed)>0)),mean_changed_fraction=float(np.mean(changed))))
        xp=extract_features(stress_frame,intervention=lambda im,row,k=kind,s=strength:perturb_image(im,row,k,s),cache_tag=f'{kind} {strength}')
        for rec in RECORDS:
            p=record_probability(rec,xp);ids,yy,pp=aggregate(sy,p,su);basep=IMAGE_BASE_PROBS[rec['id']]
            IMAGE_STRESS_PROBS[(rec['id'],kind,strength)]=pp
            score=unit_metrics(sy,p,su);base_score=macro_f1_fast(yy,basep.argmax(1))
            IMAGE_STRESS_ROWS.append(dict(model=rec['id'],kind=rec['kind'],encoding=rec['encoding'],seed=rec['seed'],
                intervention=kind,strength=strength,clean_macro_f1=base_score,
                macro_f1_change=score['macro_f1']-base_score,
                mean_unit_total_variation=float(np.mean(np.abs(pp-basep).sum(1)/2)),
                unit_prediction_flip_rate=float(np.mean(pp.argmax(1)!=basep.argmax(1))),**score))
IMAGE_STRESS_FRAME=save_table('image_stress_results.csv',IMAGE_STRESS_ROWS)
save_table('image_stress_pixel_change_audit.csv',IMAGE_STRESS_METADATA)
if not IMAGE_STRESS_FRAME.empty:
    shown=IMAGE_STRESS_FRAME[IMAGE_STRESS_FRAME.kind=='circuit'].copy()
    shown['condition']=shown.intervention+' '+shown.strength.astype(str)
    pivot=shown.groupby(['encoding','condition']).macro_f1_change.mean().unstack()
    fig,ax=plt.subplots(figsize=(10,4));im=ax.imshow(pivot,cmap='coolwarm',vmin=-.3,vmax=.3)
    ax.set_xticks(range(pivot.shape[1]),pivot.columns,rotation=55,ha='right');ax.set_yticks(range(len(pivot)),pivot.index)
    fig.colorbar(im,ax=ax,label='Macro-F1 change from clean');fig.tight_layout()
    fig.savefig(RUN_DIR/'figures'/'image_stress_heatmap.png',dpi=180);plt.show();plt.close(fig)
print('Gain/offset may be canceled by the fixed per-image robust normalization. Interpret this as a property of the entire preprocessing pipeline.')
print('The zero-border intervention preserves nonzero source pixels; it is not a validated anatomical or scanner model.')

## Cell 28 Paired uncertainty for a prespecified robustness summary

Compare the average clean-to-perturbed macro-F1 change over the fixed severity list for signed full versus signed direction. Preserve pairing across units, seeds and severities. These are secondary descriptive intervals, not additional uncorrected primary discoveries.

In [ ]:
def robustness_summary_ci(y,clean_a,stress_a,clean_b,stress_b,seed=990):
    # stress arrays: severity, fitted_seed, analysis_unit, class.
    clean_a,clean_b=align_stacks(clean_a,clean_b)
    rng=np.random.default_rng(seed);strata=[np.flatnonzero(y==j) for j in range(K) if np.any(y==j)]
    def delta(idx):
        da=[];db=[]
        for severity in range(len(stress_a)):
            aa,bb=align_stacks(stress_a[severity],stress_b[severity])
            for s in range(len(clean_a)):
                da.append(macro_f1_fast(y[idx],aa[s,idx].argmax(1))-macro_f1_fast(y[idx],clean_a[s,idx].argmax(1)))
                db.append(macro_f1_fast(y[idx],bb[s,idx].argmax(1))-macro_f1_fast(y[idx],clean_b[s,idx].argmax(1)))
        return float(np.mean(da)-np.mean(db))
    values=[delta(np.concatenate([rng.choice(s,len(s),replace=True) for s in strata])) for _ in range(CFG['bootstrap'])]
    lo,hi=np.quantile(values,[.025,.975])
    return dict(difference_in_mean_f1_change=delta(np.arange(len(y))),ci_low=float(lo),ci_high=float(hi),
                note='Secondary descriptive paired interval across fixed severities and fitted seeds; no refit variance.')
ROBUSTNESS_SUMMARIES=[]
def recs_for(enc):return sorted([r for r in RECORDS if r['kind']=='circuit' and r['encoding']==enc],key=lambda r:r['seed'])
ra,rb=recs_for('signed_full'),recs_for('signed_direction')
ca=np.stack([EVAL_PROBS[r['id']] for r in ra]);cb=np.stack([EVAL_PROBS[r['id']] for r in rb])
scales=[s for s in CFG['norm_scales'] if s!=1.]
sa=np.stack([np.stack([FEATURE_STRESS_PROBS[(r['id'],s)] for r in ra]) for s in scales])
sb=np.stack([np.stack([FEATURE_STRESS_PROBS[(r['id'],s)] for r in rb]) for s in scales])
ROBUSTNESS_SUMMARIES.append(dict(experiment='feature_norm',contrast='signed_full_minus_signed_direction',
    **robustness_summary_ci(EVAL_UNIT_Y,ca,sa,cb,sb)))
if IMAGE_STRESS_PROBS:
    chosen_conditions=[('gain',.75),('gain',1.25),('offset',-.10),('offset',.10),('bias',.15),('bias',.30),('zero_border',.20)]
    ca=np.stack([IMAGE_BASE_PROBS[r['id']] for r in ra]);cb=np.stack([IMAGE_BASE_PROBS[r['id']] for r in rb])
    sa=np.stack([np.stack([IMAGE_STRESS_PROBS[(r['id'],k,s)] for r in ra]) for k,s in chosen_conditions])
    sb=np.stack([np.stack([IMAGE_STRESS_PROBS[(r['id'],k,s)] for r in rb]) for k,s in chosen_conditions])
    ROBUSTNESS_SUMMARIES.append(dict(experiment='image_stress',contrast='signed_full_minus_signed_direction',
        **robustness_summary_ci(stress_y,ca,sa,cb,sb)))
save_table('robustness_paired_summary.csv',ROBUSTNESS_SUMMARIES)

## Cell 29 Real tumor-mask and matched-control-mask analysis

Run on all eligible supplied annotated evaluation images. Require real, aligned, nonempty, nonoverlapping tumor/control masks and area agreement within 25%. The same feathered local-fill operator applies to both masks. Expert anatomical/tissue matching must come from the annotations; code can check geometry only. Aggregate true-class probability changes by unit and export paired intervals. With no annotations, record unavailable rather than generating tumor masks.

In [ ]:
from scipy.ndimage import gaussian_filter,binary_dilation
def resolve_annotation_path(value,csv_parent):
    p=Path(value);return p if p.is_absolute() else csv_parent/p
def read_binary_mask(path,shape):
    if str(path).lower().endswith('.npy'):arr=np.load(path,allow_pickle=False)
    else:
        with Image.open(path) as image:arr=np.asarray(ImageOps.exif_transpose(image).convert('L'))
    if arr.shape!=shape:raise ValueError('Mask must match the original image dimensions exactly; automatic resizing is not used.')
    if not np.isfinite(arr).all():raise ValueError('Nonfinite mask values.')
    return arr>0
def masked_intervention(im,mask):
    ring=binary_dilation(mask,iterations=4)&~mask
    fill=float(np.median(im[ring])) if ring.any() else float(np.median(im[~mask]))
    alpha=gaussian_filter(mask.astype(float),1.)*mask
    return im*(1-alpha)+fill*alpha
MASK_RESULTS=[];MASK_FAILURES=[];MASK_SUMMARY=[]
if MASK_MANIFEST_CSV:
    mask_path=Path(MASK_MANIFEST_CSV);annotations=pd.read_csv(mask_path,dtype=str).fillna('')
    required={'relative_path','tumor_mask_path','control_mask_path'}
    if not required<=set(annotations):raise ValueError('Mask CSV requires '+str(sorted(required)))
    annotations.relative_path=annotations.relative_path.map(normalize_relative)
    if annotations.relative_path.duplicated().any():raise ValueError('Duplicate mask annotation rows.')
    lookup=annotations.set_index('relative_path');eligible=manifest.iloc[ei][manifest.iloc[ei].relative_path.isin(lookup.index)].copy()
    masks={};valid=[]
    for row in eligible.itertuples(index=False):
        try:
            ann=lookup.loc[row.relative_path];im=read_gray(row.path)
            tm=read_binary_mask(resolve_annotation_path(ann.tumor_mask_path,mask_path.parent),im.shape)
            cm=read_binary_mask(resolve_annotation_path(ann.control_mask_path,mask_path.parent),im.shape)
            if not tm.any() or not cm.any() or (tm&cm).any():raise ValueError('Tumor/control masks must be nonempty and nonoverlapping.')
            if not .8<=cm.sum()/tm.sum()<=1.25:raise ValueError('Control-mask area differs by more than the prespecified 25% tolerance.')
            if row.label=='notumor':raise ValueError('A tumor mask was supplied for a no-tumor label.')
            masks[row.relative_path]=(tm,cm);valid.append(row.relative_path)
        except Exception as exc:MASK_FAILURES.append(dict(relative_path=row.relative_path,error=str(exc)))
    eligible=eligible[eligible.relative_path.isin(valid)].copy()
    if len(eligible):
        mi=eligible.index.to_numpy();mx=X_ALL[mi];my=Y_ALL[mi];mu=U_ALL[mi]
        xt=extract_features(eligible,lambda im,row:masked_intervention(im,masks[row.relative_path][0]),cache_tag='Tumor-mask sensitivity')
        xc=extract_features(eligible,lambda im,row:masked_intervention(im,masks[row.relative_path][1]),cache_tag='Control-mask sensitivity')
        for rec in RECORDS:
            p0=record_probability(rec,mx);pt=record_probability(rec,xt);pc=record_probability(rec,xc)
            ids,yy,bp=aggregate(my,p0,mu);_,_,tp=aggregate(my,pt,mu);_,_,cp=aggregate(my,pc,mu)
            tumor_drop=bp[np.arange(len(yy)),yy]-tp[np.arange(len(yy)),yy]
            control_drop=bp[np.arange(len(yy)),yy]-cp[np.arange(len(yy)),yy];delta=tumor_drop-control_drop
            rng=np.random.default_rng(710);boot=np.array([rng.choice(delta,len(delta),replace=True).mean() for _ in range(CFG['bootstrap'])])
            lo,hi=np.quantile(boot,[.025,.975])
            MASK_SUMMARY.append(dict(model=rec['id'],n_units=len(ids),n_images=len(eligible),unit_kind=UNIT_KIND,
                mean_tumor_minus_control_drop=float(delta.mean()),ci_low=float(lo),ci_high=float(hi),
                note='Exploratory sensitivity. Geometric checks do not verify tissue/location matching; expert annotations are required.'))
            for unit,td,cd in zip(ids,tumor_drop,control_drop):MASK_RESULTS.append(dict(model=rec['id'],analysis_unit=unit,tumor_drop=float(td),control_drop=float(cd)))
        STATUS['mask_analysis']='completed_with_supplied_masks'
    else:STATUS['mask_analysis']='no_valid_evaluation_mask_pairs'
else:
    STATUS['mask_analysis']='unavailable: dataset supplies no tumor/control annotations'
    print('Mask analysis skipped: no mask CSV supplied. No artificial tumor locations or masks are generated.')
save_table('mask_sensitivity_by_unit.csv',pd.DataFrame(MASK_RESULTS,columns=['model','analysis_unit','tumor_drop','control_drop']))
save_table('mask_sensitivity_summary.csv',pd.DataFrame(MASK_SUMMARY))
save_table('mask_validation_failures.csv',pd.DataFrame(MASK_FAILURES,columns=['relative_path','error']))

## Cell 30 Optional external cohort with fixed models and overlap auditing

Use the same frozen image transform, backbone, preprocessing, chosen model and temperature. Audit exact/near overlap against every image in the original Kaggle collection and remove overlapping external units. Verified external patient IDs and independent provenance require original records. If an external cohort is supplied, evaluate the same three contrasts as a separate prespecified validation family. No external data are downloaded or invented automatically.

In [ ]:
EXTERNAL_RESULTS=[];EXTERNAL_PREDICTIONS=[];EXTERNAL_INFERENCE=[]
if EXTERNAL_DATA_ROOT:
    external_raw,external_bad=scan_images(EXTERNAL_DATA_ROOT)
    external_clean,external_quarantine,external_cross,external_edges,external_audit=group_and_clean(external_raw,EXTERNAL_PATIENT_CSV)
    external_unit_kind=external_audit['unit_kind'];external_overlap=[];external_tree=BKTree()
    training_ph=[int(s,16) for s in raw_manifest.phash];training_dh=[int(s,16) for s in raw_manifest.dhash]
    training_aspect=(raw_manifest.width/raw_manifest.height).to_numpy();known_hashes=set(raw_manifest.pixel_sha256)
    for i,h in enumerate(training_ph):external_tree.add(h,i)
    for row in external_clean.itertuples(index=False):
        exact=row.pixel_sha256 in known_hashes;near=False
        for j in external_tree.query(int(row.phash,16),CFG['phash_radius']):
            if (int(row.dhash,16)^training_dh[j]).bit_count()<=CFG['dhash_radius'] and abs(math.log((row.width/row.height)/training_aspect[j]))<=CFG['aspect_tolerance']:
                near=True;break
        if exact or near:external_overlap.append(dict(relative_path=row.relative_path,analysis_unit=row.analysis_unit,exact=exact,near=near))
    overlap_units={r['analysis_unit'] for r in external_overlap}
    if PATIENT_MANIFEST_CSV and EXTERNAL_PATIENT_CSV:
        overlap_units|=set(raw_manifest.get('patient_id',pd.Series(dtype=str)))&set(external_clean.patient_id)
        # raw_manifest has no attached metadata; compare the attached retained development IDs as well.
        overlap_units|=set(clean_manifest.patient_id)&set(external_clean.patient_id)
    external_clean=external_clean[~external_clean.analysis_unit.isin(overlap_units)].copy().reset_index(drop=True)
    save_table('external_overlap_audit.csv',pd.DataFrame(external_overlap,columns=['relative_path','analysis_unit','exact','near']))
    save_table('external_decode_failures.csv',pd.DataFrame(external_bad,columns=['relative_path','error']))
    save_table('external_quarantine.csv',external_quarantine);save_table('external_study_manifest.csv',external_clean)
    if external_clean.empty:
        STATUS['external_evaluation']='no eligible images after overlap/quality auditing'
    else:
        ex=extract_features(external_clean,cache_tag='External frozen features')
        eyy=external_clean.y.to_numpy();euu=external_clean.analysis_unit.to_numpy(dtype=str);ex_probs={}
        external_ids,external_y,_=aggregate(eyy,np.ones((len(eyy),K))/K,euu)
        for rec in RECORDS:
            p=record_probability(rec,ex);ids,yy,pp=aggregate(eyy,p,euu);ex_probs[rec['id']]=pp
            EXTERNAL_RESULTS.append(dict(model=rec['id'],kind=rec['kind'],encoding=rec['encoding'],seed=rec['seed'],
                                         unit_kind=external_unit_kind,**unit_metrics(eyy,p,euu)))
            for unit,label,prob in zip(ids,yy,pp):
                EXTERNAL_PREDICTIONS.append(dict(model=rec['id'],analysis_unit=unit,y=int(label),**{f'p{j}':float(v) for j,v in enumerate(prob)}))
        if set(external_y)==set(range(K)):
            def external_stack(encoding,kind):
                rr=sorted([r for r in RECORDS if r['encoding']==encoding and r['kind']==kind],key=lambda r:r['seed'])
                return np.stack([ex_probs[r['id']] for r in rr])
            for j,(name,a,b) in enumerate(contrasts):
                EXTERNAL_INFERENCE.append(dict(contrast=name,**paired_inference(external_y,external_stack(*a),external_stack(*b),
                                                                               seed=200+j,unit_kind=external_unit_kind)))
            order=np.argsort([r['permutation_p_two_sided'] for r in EXTERNAL_INFERENCE]);previous=0.
            for rank,j in enumerate(order):
                previous=min(1.,max(previous,(len(order)-rank)*EXTERNAL_INFERENCE[j]['permutation_p_two_sided']))
                EXTERNAL_INFERENCE[j]['holm_adjusted_p']=previous
        else:print('External cohort lacks at least one class. Four-class confirmatory contrasts are not computed.')
        STATUS['external_evaluation']='completed_with_'+external_unit_kind
        STATUS['external_provenance']='Independent collection must be documented by the investigator; overlap auditing cannot prove provenance.'
        save_json('external_audit.json',external_audit)
else:
    STATUS['external_evaluation']='unavailable: no separate compatible cohort supplied'
    print('External evaluation skipped. Kaggle Testing is not relabeled as an independent cohort.')
save_table('external_evaluation_metrics.csv',pd.DataFrame(EXTERNAL_RESULTS))
save_table('external_unit_predictions.csv',pd.DataFrame(EXTERNAL_PREDICTIONS))
save_table('external_paired_inference.csv',pd.DataFrame(EXTERNAL_INFERENCE))

## Cell 31 Traceability and final reproducibility checks

Map every document requirement to its code/output. Recheck split isolation, finite primary metrics and exact equality. Export probability arrays so secondary stress results can be independently reanalyzed. Missing evidence remains explicit in study_status.json.

In [ ]:
requirements=[
    ('Document direction 1','Sign/norm/direction collision audit; five common-register encoders; validation-only scale selection','synthetic_information_audit.csv'),
    ('Document direction 2','Exact class matrices; unrestricted same-information quadratic lift; low-rank and shared-observable unitary controls','real_image_exact_equivalence.csv'),
    ('Optimization diagnosis','Realizable teacher probabilities, multiple teachers/seeds and two optimizers','realizable_teacher_recovery.csv'),
    ('Document direction 3','Fixed-direction norm interventions and image-space acquisition stress tests','image_stress_results.csv'),
    ('Tumor localization','All supplied eligible tumor/control mask pairs; explicit unavailable status when masks are absent','mask_sensitivity_summary.csv'),
    ('Document direction 4','Repeated-seed fixed-hyperparameter learning curves; preprocessing refitted for each subset','learning_curve_evaluation.csv'),
    ('Data integrity','Decoded pixel hashes, two perceptual hashes, graph grouping and cross-boundary audit','duplicate_audit.json'),
    ('Refitting variance','Nested grouped development folds; evaluation partition never used for model selection','nested_grouped_development_cv.csv'),
    ('Reliability','Separate temperature partition; raw/calibrated NLL, Brier, ECE and AUC','evaluation_metrics.csv'),
    ('Inference','Paired unit bootstrap/permutation, fixed seeds and Holm correction','primary_paired_inference.csv'),
    ('External validation','Optional separate cohort; frozen pipeline; exact/near overlap audit','external_evaluation_metrics.csv'),
]
save_table('document_requirements_traceability.csv',pd.DataFrame(requirements,columns=['requirement','implementation','artifact']))
assert all((RUN_DIR/name).exists() for _,_,name in requirements)
assert not (manifest.groupby('analysis_unit').partition.nunique()>1).any()
assert max(r['maximum_logit_error'] for r in PARITY_ROWS)<1e-9
assert np.isfinite(METRICS_FRAME.macro_f1).all()
save_json('study_status.json',STATUS)
save_json('validation_summary.json',dict(numerical_checks_passed=True,gradient_error=gradient_error,
    maximum_real_data_equivalence_error=max(r['maximum_logit_error'] for r in PARITY_ROWS),
    raw_images=len(raw_manifest),analyzed_images=len(manifest),evaluation_units=len(EVAL_UNIT_Y),unit_kind=UNIT_KIND,
    status=STATUS,versions=VERSIONS,profile=PROFILE,
    limitation='Numerical checks do not prove model convergence, patient independence or clinical generalization.'))
# Full probability arrays support independent reanalysis of secondary stress tests.
np.savez_compressed(RUN_DIR/'feature_stress_unit_probabilities.npz',
    analysis_unit=EVAL_UNIT_IDS,y=EVAL_UNIT_Y,
    **{name+'__scale_'+str(scale):p for (name,scale),p in FEATURE_STRESS_PROBS.items()})
if IMAGE_STRESS_PROBS:
    np.savez_compressed(RUN_DIR/'image_stress_unit_probabilities.npz',analysis_unit=stress_ids,y=stress_y,
        **{name+'__'+kind+'_'+str(strength):p for (name,kind,strength),p in IMAGE_STRESS_PROBS.items()})

## Cell 32 Generate a readable results report and download archive

Create RESULTS_REPORT.md, all CSVs, checkpoints, PNG figures, hashes and a ZIP. In Colab, set auto_download_results=True to trigger a download, or use the Files pane. The report records the profile and limits so smoke/pilot output cannot be presented as a completed clinical validation.

In [ ]:
report=[]
report.append('# Quantum MRI encoding and robustness results\n')
report.append(f'Run `{RUN_ID}`; profile `{PROFILE}`; analysis unit `{UNIT_KIND}`.\n')
report.append('These results are simulator experiments on the downloaded cohort. Dataset folders and duplicate groups do not establish patient independence.\n')
report.append('## Dataset and audit\n')
report.append(json.dumps(duplicate_audit,indent=2)+'\n')
report.append('## Primary paired comparisons\n')
report.append(PRIMARY_FRAME[['contrast','delta_macro_f1','ci_low','ci_high','holm_adjusted_p']].to_string(index=False)+'\n')
report.append('Intervals are conditional on the fitted seed set and are pointwise; Holm adjustment applies to the three permutation p-values. The margin is an engineering analysis choice, not a clinically established threshold.\n')
report.append('## Available evidence\n'+json.dumps(STATUS,indent=2)+'\n')
report.append('## Interpretation limits\n')
report.append('The norm encoders and circuit architecture are fully specified reference implementations. They are not a verified reproduction of the original manuscript Coherent5/Passive5.\n')
report.append('Unrestricted quadratic models have more freedom than shallow circuits; a difference alone does not separate representation from optimization. Teacher and matrix-constrained controls provide additional diagnostic evidence.\n')
report.append('The full-unitary control is a dense classical simulation preserving the shared-observable construction. Its runtime and parameter count do not represent an implementable shallow quantum circuit.\n')
report.append('A smoke or pilot profile uses reduced data and optimization budgets. Learning curves can skip tiny subsets that cannot support PCA16; skipped cases are exported.\n')
report.append('Mask/control geometry checks do not verify anatomical or tissue matching. Scanner stress tests are synthetic image interventions. External provenance and verified patient identity require original metadata.\n')
report.append('Nested development folds describe refitting variation. Folds and seeds must not be counted as additional independent patients. Nonsignificance alone does not establish equivalence.\n')
report.append('## Reproducibility\nProtocol SHA256: '+PROTOCOL_SHA+'\n\n'+json.dumps(VERSIONS,indent=2)+'\n')
(RUN_DIR/'RESULTS_REPORT.md').write_text('\n\n'.join(report))
save_json('evaluation_complete.json',dict(protocol_sha256=PROTOCOL_SHA,completed_utc=datetime.now(timezone.utc).isoformat()))
archive_base=WORK_ROOT/('bqml_results_'+RUN_ID)
RESULTS_ZIP=Path(shutil.make_archive(str(archive_base),'zip',root_dir=RUN_DIR))
print('RESULTS ARCHIVE:',RESULTS_ZIP)
print('All requested experiments have runnable cells. Evidence-dependent mask/patient/external stages report unavailable when their inputs are absent.')
display(pd.DataFrame([STATUS]))
if CFG['auto_download_results']:
    try:
        from google.colab import files
        files.download(str(RESULTS_ZIP))
    except ImportError:print('Outside Colab: copy the results ZIP from the path above.')

## Scientific references and source requirements

* User-supplied requirements: bqml_3more_innovations.docx.
* [Kaggle Brain Tumor MRI dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset/data).
* [Schuld et al. Circuit-centric quantum classifiers](https://arxiv.org/abs/1804.00633): amplitude encoding and nonzero padding have prior art.
* [Schreiber et al. Classical surrogates for quantum learning models](https://arxiv.org/abs/2206.11740).
* [Bowles et al. Better than classical](https://arxiv.org/abs/2403.07059).
* [Torchvision ResNet18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html).
* [scikit-learn leakage and preprocessing](https://scikit-learn.org/stable/common_pitfalls.html).
* [scikit-learn grouped cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html).

The proposed contribution is the controlled empirical relationship between encoding
information, circuit restrictions and robustness. Running this notebook does not
establish that a method is novel; assess priority against the final related work.

## Validation record for this deliverable

The notebook was tested outside Colab using the supplied real Kaggle images and a CPU runtime. This validates the recorded execution scope, not every possible Colab package version, network condition or full-profile optimization outcome.

```json
{
  "validated_on": "2026-10-16",
  "execution_method": "Exact notebook code cells executed sequentially in a Python process outside Colab. Socket restrictions prevented launching a Jupyter kernel.",
  "real_kaggle_pilot": {
    "code_cells_executed": 32,
    "code_cell_errors": 0,
    "downloaded_images_audited": 7200,
    "images_in_reduced_study": 1259,
    "evaluation_duplicate_proxy_units": 320,
    "stochastic_seeds": [
      0,
      1
    ],
    "elapsed_seconds": 321.29
  },
  "optional_input_branch_validation": {
    "scope": "Synthetic numeric-image fixtures only; not clinical patient or mask evidence.",
    "code_cells_executed": 32,
    "code_cell_errors": 0,
    "patient_metadata_path": "passed",
    "tumor_and_control_mask_path": "passed",
    "external_cohort_path": "passed",
    "encoding_scale_search": [
      0.5,
      1.0,
      2.0
    ]
  },
  "numerical_checks": {
    "maximum_gradient_absolute_error": 2.3728324860528005e-11,
    "maximum_real_image_quadratic_logit_error": 1.9539925233402755e-14,
    "state_information_unitarity_dephasing_and_direction_scale_invariance": "passed"
  },
  "not_executed_here": [
    "live Google Colab runtime",
    "GPU feature extraction",
    "full profile training with all seeds and full search"
  ],
  "validated_versions": {
    "numpy": "2.3.5",
    "scipy": "1.17.0",
    "pandas": "2.2.3",
    "sklearn": "1.8.0",
    "torch": "2.14.1+cpu",
    "torchvision": "0.29.1+cpu"
  },
  "code_cells_sha256": "c0150391fd966854c8e93ec5a9d1ff43aa8c98c6de3d2ad903a0a06c926a8473"
}
```